# H23 protocol v4 observation capture

Run interactively on Kaggle only after local review. This notebook performs the
official all-wheel pip invocation recipe and writes a bounded observation
archive. It excludes CUTLASS wheels and restores stripped cu128 version markers;
it does not delete pre-existing cutlass.pth files or claim whole-cell equivalence.
It never imports a target package or runs a model. A bootstrap failure is
reported as an observation; it does not trigger an automatic fallback.
Use a fresh kernel: an existing tools package causes a fixed failure before the
embedded package is imported.

The archive does not prove Kaggle origin, successful installation, runtime
functionality, or hidden scorer identity. Independent import validates only
the supplied evidence. Human run attestation is a separate project process.


In [ ]:
import json
import pathlib
import sys
import tempfile

if 'tools' in sys.modules:
    raise RuntimeError('H23_V4_REQUIRES_FRESH_KERNEL')
H23_RUNTIME_SOURCES = json.loads('{"tools/__init__.py": "", "tools/h23_v4/__init__.py": "\\"\\"\\"H23 observation archive protocol v4; no legacy compatibility.\\"\\"\\"\\n", "tools/h23_v4/archive.py": "\\"\\"\\"Bounded v4 ZIP parsing: one manifest and content-addressed payload bytes.\\"\\"\\"\\n\\nfrom dataclasses import dataclass\\nimport hashlib\\nimport io\\nimport json\\nimport re\\nimport stat\\nimport struct\\nimport zipfile\\nimport zlib\\n\\nfrom .schema import MAX_MANIFEST_BYTES, PolicyError, strict_json\\n\\nMAX_ARCHIVE_BYTES = 128 * 1024 * 1024\\nMAX_EXPANDED_BYTES = 256 * 1024 * 1024\\nMAX_MEMBER_BYTES = 32 * 1024 * 1024\\nMAX_PAYLOADS = 8192\\nMAX_MEMBERS = MAX_PAYLOADS + 1\\n_BLOB = re.compile(r\\"blobs/([0-9a-f]{64})\\\\Z\\")\\n_BLOB_BYTES = re.compile(rb\\"blobs/[0-9a-f]{64}\\\\Z\\")\\n_EOCD = struct.Struct(\\"<4s4H2LH\\")\\n_CENTRAL = struct.Struct(\\"<4s6H3L5H2L\\")\\n_LOCAL = struct.Struct(\\"<4s5H3L2H\\")\\n\\n\\n@dataclass(frozen=True)\\nclass ArchiveContents:\\n    manifest: dict\\n    blobs: dict\\n    archive_sha256: str\\n\\n\\ndef _references(manifest):\\n    if not isinstance(manifest, dict):\\n        raise PolicyError(\\"SCHEMA_FIELDS\\")\\n    references = manifest.get(\\"payload_references\\")\\n    if not isinstance(references, list) or len(references) > MAX_PAYLOADS:\\n        raise PolicyError(\\"PAYLOAD_REFERENCES\\")\\n    if any(type(item) is not str or re.fullmatch(r\\"[0-9a-f]{64}\\", item) is None for item in references):\\n        raise PolicyError(\\"PAYLOAD_REFERENCES\\")\\n    if references != sorted(set(references)):\\n        raise PolicyError(\\"PAYLOAD_REFERENCES\\")\\n    return set(references)\\n\\n\\ndef _preflight_archive(snapshot):\\n    \\"\\"\\"Check the closed non-ZIP64 envelope before ZipFile allocates members.\\n\\n    The only supported topology is contiguous local headers/data, an equally\\n    ordered contiguous central directory, and a final 22-byte, comment-free\\n    EOCD. The walk is capped by the independently enforced member limit. Raw\\n    local headers are checked separately: ZipInfo exposes only central extras.\\n    \\"\\"\\"\\n    if len(snapshot) < _EOCD.size:\\n        raise PolicyError(\\"ARCHIVE_ENVELOPE\\")\\n    end_offset = len(snapshot) - _EOCD.size\\n    (signature, disk, central_disk, disk_count, count, central_size,\\n     central_offset, comment_size) = _EOCD.unpack_from(snapshot, end_offset)\\n    if (signature != b\\"PK\\\\x05\\\\x06\\" or comment_size or disk or central_disk\\n            or disk_count != count):\\n        raise PolicyError(\\"ARCHIVE_ENVELOPE\\")\\n    if not 1 <= count <= MAX_MEMBERS:\\n        raise PolicyError(\\"ARCHIVE_MEMBERS\\")\\n    if (central_offset + central_size != end_offset\\n            or central_size < count * _CENTRAL.size):\\n        raise PolicyError(\\"ARCHIVE_ENVELOPE\\")\\n\\n    cursor, next_local, expanded = central_offset, 0, 0\\n    names = set()\\n    for _ in range(count):\\n        if cursor + _CENTRAL.size > end_offset:\\n            raise PolicyError(\\"ARCHIVE_ENVELOPE\\")\\n        (signature, _created, needed, flags, method, time, date, crc,\\n         compressed_size, size, name_size, extra_size, member_comment_size,\\n         member_disk, _internal, external, local_offset) = _CENTRAL.unpack_from(snapshot, cursor)\\n        if (signature != b\\"PK\\\\x01\\\\x02\\" or extra_size or member_comment_size\\n                or member_disk):\\n            raise PolicyError(\\"ARCHIVE_ENVELOPE\\")\\n        if name_size not in (13, 70):\\n            raise PolicyError(\\"ARCHIVE_MEMBER_NAME\\")\\n        name_start = cursor + _CENTRAL.size\\n        cursor = name_start + name_size\\n        if cursor > end_offset:\\n            raise PolicyError(\\"ARCHIVE_ENVELOPE\\")\\n        name = snapshot[name_start:cursor]\\n        if name != b\\"manifest.json\\" and _BLOB_BYTES.fullmatch(name) is None:\\n            raise PolicyError(\\"ARCHIVE_MEMBER_NAME\\")\\n        if name in names:\\n            raise PolicyError(\\"ARCHIVE_DUPLICATE_MEMBER\\")\\n        names.add(name)\\n        if stat.S_IFMT(external >> 16) not in (0, stat.S_IFREG):\\n            raise PolicyError(\\"ARCHIVE_MEMBER_TYPE\\")\\n        if flags & 1 or method not in (zipfile.ZIP_STORED, zipfile.ZIP_DEFLATED):\\n            raise PolicyError(\\"ARCHIVE_COMPRESSION\\")\\n        # ASCII names need no flags. Descriptors, strong encryption, ZIP64 and\\n        # other extraction profiles have no representation in this contract.\\n        if flags or needed not in (10, 20) or (method == zipfile.ZIP_DEFLATED and needed != 20):\\n            raise PolicyError(\\"ARCHIVE_ENVELOPE\\")\\n        limit = MAX_MANIFEST_BYTES if name == b\\"manifest.json\\" else MAX_MEMBER_BYTES\\n        if size > limit or compressed_size > MAX_ARCHIVE_BYTES:\\n            raise PolicyError(\\"ARCHIVE_MEMBER_SIZE\\")\\n        if method == zipfile.ZIP_STORED and compressed_size != size:\\n            raise PolicyError(\\"ARCHIVE_ENVELOPE\\")\\n        expanded += size\\n        if expanded > MAX_EXPANDED_BYTES:\\n            raise PolicyError(\\"ARCHIVE_EXPANDED_SIZE\\")\\n\\n        if local_offset != next_local or local_offset + _LOCAL.size > central_offset:\\n            raise PolicyError(\\"ARCHIVE_ENVELOPE\\")\\n        (local_signature, local_needed, local_flags, local_method, local_time,\\n         local_date, local_crc, local_compressed, local_size,\\n         local_name_size, local_extra_size) = _LOCAL.unpack_from(snapshot, local_offset)\\n        if (local_signature != b\\"PK\\\\x03\\\\x04\\" or local_extra_size\\n                or (local_needed, local_flags, local_method, local_time, local_date,\\n                    local_crc, local_compressed, local_size, local_name_size)\\n                != (needed, flags, method, time, date, crc, compressed_size, size, name_size)):\\n            raise PolicyError(\\"ARCHIVE_ENVELOPE\\")\\n        local_name_start = local_offset + _LOCAL.size\\n        payload_start = local_name_start + name_size\\n        next_local = payload_start + compressed_size\\n        if next_local > central_offset or snapshot[local_name_start:payload_start] != name:\\n            raise PolicyError(\\"ARCHIVE_ENVELOPE\\")\\n\\n    if cursor != end_offset or next_local != central_offset:\\n        raise PolicyError(\\"ARCHIVE_ENVELOPE\\")\\n    if b\\"manifest.json\\" not in names:\\n        raise PolicyError(\\"ARCHIVE_MANIFEST\\")\\n    return count\\n\\n\\ndef _member_bytes(snapshot, member, limit):\\n    \\"\\"\\"Decode exactly the preflighted compressed extent, including its end.\\n\\n    ZipExtFile can stop at the reported file size and ignore unused deflate\\n    bytes or a missing stream terminator. Those bytes have no V4 payload role.\\n    A raw decoder provides a bounded independent check of the entire extent.\\n    \\"\\"\\"\\n    start = member.header_offset + _LOCAL.size + len(member.filename.encode(\\"ascii\\"))\\n    end = start + member.compress_size\\n    if member.compress_type == zipfile.ZIP_STORED:\\n        data = snapshot[start:end]\\n    else:\\n        decoder = zlib.decompressobj(-zlib.MAX_WBITS)\\n        data = decoder.decompress(memoryview(snapshot)[start:end], limit + 1)\\n        if len(data) > limit:\\n            raise PolicyError(\\"ARCHIVE_MEMBER_SIZE\\")\\n        if not decoder.eof or decoder.unused_data or decoder.unconsumed_tail:\\n            raise PolicyError(\\"ARCHIVE_ENVELOPE\\")\\n    if len(data) > limit or len(data) != member.file_size:\\n        raise PolicyError(\\"ARCHIVE_MEMBER_SIZE\\")\\n    if zlib.crc32(data) != member.CRC:\\n        raise PolicyError(\\"ARCHIVE_MALFORMED\\")\\n    return data\\n\\n\\ndef read_archive(snapshot):\\n    \\"\\"\\"Hash and parse precisely the same immutable, bounded byte snapshot.\\n\\n    This enforces the archive envelope. The importer separately validates the\\n    manifest\'s typed observations and the relationships between them.\\n    \\"\\"\\"\\n    if type(snapshot) is not bytes or len(snapshot) > MAX_ARCHIVE_BYTES:\\n        raise PolicyError(\\"ARCHIVE_SIZE\\")\\n    count = _preflight_archive(snapshot)\\n    try:\\n        with zipfile.ZipFile(io.BytesIO(snapshot)) as archive:\\n            members = archive.infolist()\\n            if len(members) != count or not 1 <= len(members) <= MAX_MEMBERS:\\n                raise PolicyError(\\"ARCHIVE_MEMBERS\\")\\n            names = [member.filename for member in members]\\n            if len(set(names)) != len(names):\\n                raise PolicyError(\\"ARCHIVE_DUPLICATE_MEMBER\\")\\n            if names.count(\\"manifest.json\\") != 1:\\n                raise PolicyError(\\"ARCHIVE_MANIFEST\\")\\n            expanded = 0\\n            for member in members:\\n                if member.orig_filename != member.filename:\\n                    raise PolicyError(\\"ARCHIVE_MEMBER_NAME\\")\\n                if member.filename != \\"manifest.json\\" and _BLOB.fullmatch(member.filename) is None:\\n                    raise PolicyError(\\"ARCHIVE_MEMBER_NAME\\")\\n                kind = stat.S_IFMT(member.external_attr >> 16)\\n                if member.is_dir() or kind not in (0, stat.S_IFREG):\\n                    raise PolicyError(\\"ARCHIVE_MEMBER_TYPE\\")\\n                if member.flag_bits & 1 or member.compress_type not in (zipfile.ZIP_STORED, zipfile.ZIP_DEFLATED):\\n                    raise PolicyError(\\"ARCHIVE_COMPRESSION\\")\\n                limit = MAX_MANIFEST_BYTES if member.filename == \\"manifest.json\\" else MAX_MEMBER_BYTES\\n                if member.file_size > limit or member.compress_size > MAX_ARCHIVE_BYTES:\\n                    raise PolicyError(\\"ARCHIVE_MEMBER_SIZE\\")\\n                expanded += member.file_size\\n                if expanded > MAX_EXPANDED_BYTES:\\n                    raise PolicyError(\\"ARCHIVE_EXPANDED_SIZE\\")\\n            raw_members = {}\\n            for member in members:\\n                limit = MAX_MANIFEST_BYTES if member.filename == \\"manifest.json\\" else MAX_MEMBER_BYTES\\n                raw_members[member.filename] = _member_bytes(snapshot, member, limit)\\n    except PolicyError:\\n        raise\\n    except (zipfile.BadZipFile, zipfile.LargeZipFile, RuntimeError, OSError, EOFError, ValueError,\\n            NotImplementedError, zlib.error):\\n        raise PolicyError(\\"ARCHIVE_MALFORMED\\") from None\\n    manifest = strict_json(raw_members.pop(\\"manifest.json\\"))\\n    references = _references(manifest)\\n    blobs = {}\\n    for name, data in raw_members.items():\\n        digest = name[6:]\\n        if hashlib.sha256(data).hexdigest() != digest:\\n            raise PolicyError(\\"BLOB_HASH\\")\\n        blobs[digest] = data\\n    if references - blobs.keys():\\n        raise PolicyError(\\"BLOB_MISSING\\")\\n    if blobs.keys() - references:\\n        raise PolicyError(\\"BLOB_ORPHAN\\")\\n    return ArchiveContents(manifest, blobs, hashlib.sha256(snapshot).hexdigest())\\n\\n\\nparse_archive = read_archive\\n\\n\\ndef _member(name, data):\\n    info = zipfile.ZipInfo(name, date_time=(1980, 1, 1, 0, 0, 0))\\n    info.create_system = 3\\n    info.create_version = 20\\n    info.extract_version = 20\\n    info.flag_bits = 0\\n    info.extra = b\\"\\"\\n    info.comment = b\\"\\"\\n    info.internal_attr = 0\\n    info.external_attr = (stat.S_IFREG | 0o600) << 16\\n    info.compress_type = zipfile.ZIP_DEFLATED\\n    return info, data\\n\\n\\ndef build_archive(manifest, blobs):\\n    \\"\\"\\"Create deterministic envelope bytes, rejecting invalid blob commitments.\\"\\"\\"\\n    if not isinstance(blobs, dict):\\n        raise PolicyError(\\"PAYLOAD_REFERENCES\\")\\n    try:\\n        raw_manifest = json.dumps(manifest, sort_keys=True, separators=(\\",\\", \\":\\"), ensure_ascii=False,\\n                                  allow_nan=False).encode(\\"utf-8\\")\\n    except (TypeError, ValueError, OverflowError):\\n        raise PolicyError(\\"JSON_TYPE\\") from None\\n    if len(blobs) > MAX_PAYLOADS:\\n        raise PolicyError(\\"ARCHIVE_MEMBERS\\")\\n    if len(raw_manifest) > MAX_MANIFEST_BYTES:\\n        raise PolicyError(\\"ARCHIVE_MEMBER_SIZE\\")\\n    total = len(raw_manifest)\\n    for digest, data in blobs.items():\\n        if type(digest) is not str or re.fullmatch(r\\"[0-9a-f]{64}\\", digest) is None or type(data) is not bytes:\\n            raise PolicyError(\\"BLOB_HASH\\")\\n        if len(data) > MAX_MEMBER_BYTES or hashlib.sha256(data).hexdigest() != digest:\\n            raise PolicyError(\\"BLOB_HASH\\")\\n        total += len(data)\\n        if total > MAX_EXPANDED_BYTES:\\n            raise PolicyError(\\"ARCHIVE_EXPANDED_SIZE\\")\\n    output = io.BytesIO()\\n    with zipfile.ZipFile(output, \\"w\\", compression=zipfile.ZIP_DEFLATED,\\n                         compresslevel=9, allowZip64=False) as archive:\\n        archive.comment = b\\"\\"\\n        archive.writestr(*_member(\\"manifest.json\\", raw_manifest), compresslevel=9)\\n        for digest, data in sorted(blobs.items()):\\n            archive.writestr(*_member(\\"blobs/\\" + digest, data), compresslevel=9)\\n            if output.tell() > MAX_ARCHIVE_BYTES:\\n                raise PolicyError(\\"ARCHIVE_SIZE\\")\\n    result = output.getvalue()\\n    read_archive(result)\\n    return result\\n", "tools/h23_v4/capture.py": "\\"\\"\\"Produce bounded, untrusted observations without claiming installation or origin.\\n\\nThe only subprocess is the intentional official pip bootstrap. No target module\\nis imported, and arbitrary metadata, direct URLs, environment values and pip\\noutput are never archive payloads.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport datetime as dt\\nimport hashlib\\nimport os\\nimport stat\\nimport subprocess\\nimport sys\\nimport sysconfig\\nimport tempfile\\nimport zipfile\\nfrom dataclasses import dataclass, field\\nfrom pathlib import Path\\n\\nfrom tools.h23_v4.archive import MAX_EXPANDED_BYTES, MAX_PAYLOADS, build_archive\\nfrom tools.h23_v4.filesystem import anchor_directory, read_regular\\nfrom tools.h23_v4.metadata_policy import (parse_console_scripts, parse_metadata,\\n                                         validate_wheel_dist_info, wheel_filename_identity,\\n                                         wheel_installation_filename)\\nfrom tools.h23_v4.record_policy import classify_record_rows, payload_required, validate_interpreter_roots\\nfrom tools.h23_v4.schema import PolicyError, canonical_json, validate_manifest\\n\\n\\nTARGETS = (\\n    \\"swegemma\\", \\"adk-submission\\", \\"adk-eval-core\\", \\"google-adk\\", \\"google-genai\\",\\n    \\"litellm\\", \\"vllm\\", \\"transformers\\",\\n)\\nRUNTIME_FILES = (\\n    \\"tools/__init__.py\\", \\"tools/h23_v4/__init__.py\\", \\"tools/h23_v4/schema.py\\",\\n    \\"tools/h23_v4/metadata_policy.py\\", \\"tools/h23_v4/record_policy.py\\",\\n    \\"tools/h23_v4/filesystem.py\\", \\"tools/h23_v4/archive.py\\", \\"tools/h23_v4/capture.py\\",\\n)\\nOFFICIAL_ENVIRONMENT = {\\n    \\"LITELLM_LOCAL_MODEL_COST_MAP\\": \\"True\\", \\"TRANSFORMERS_NO_TF\\": \\"1\\",\\n    \\"VLLM_WORKER_MULTIPROC_METHOD\\": \\"spawn\\", \\"VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS\\": \\"1\\",\\n    \\"VLLM_ENGINE_READY_TIMEOUT_S\\": \\"1200\\", \\"VLLM_NO_USAGE_STATS\\": \\"1\\",\\n    \\"OTEL_SDK_DISABLED\\": \\"true\\", \\"PYTORCH_CUDA_ALLOC_CONF\\": \\"expandable_segments:True\\",\\n}\\nMAX_PAYLOAD_BYTES = 32 * 1024 * 1024\\nMAX_WHEEL_BYTES = 16 * 2**30\\n\\n\\ndef _site_roots():\\n    return tuple(sorted({Path(sysconfig.get_path(key)).resolve() for key in (\\"purelib\\", \\"platlib\\")}))\\n\\n\\ndef _installation_scheme():\\n    # Python names the default virtualenv scheme \\"venv\\" on POSIX. This sole\\n    # documented alias changes the label, never the observed roots or command.\\n    observed = sysconfig.get_default_scheme()\\n    return \\"posix_venv\\" if observed == \\"venv\\" else observed\\n\\n\\n@dataclass(frozen=True)\\nclass CaptureConfig:\\n    wheelhouse: Path = Path(\\"/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse\\")\\n    output_directory: Path = Path(\\"/kaggle/working\\")\\n    site_roots: tuple[Path, ...] = field(default_factory=_site_roots)\\n    prefix: Path = field(default_factory=lambda: Path(sys.prefix).resolve())\\n    scripts_directory: Path = field(default_factory=lambda: Path(sysconfig.get_path(\\"scripts\\")).resolve())\\n    executable: Path = field(default_factory=lambda: Path(sys.executable).absolute())\\n    python_version: tuple[int, int, int] = field(default_factory=lambda: tuple(sys.version_info[:3]))\\n    scheme: str = field(default_factory=_installation_scheme)\\n    execute_bootstrap: bool = True\\n    executed_notebook: Path | None = None\\n\\n\\ndef capture_program_sha256():\\n    project = Path(__file__).absolute().parents[2]\\n    commitments = {name: hashlib.sha256((project / name).read_bytes()).hexdigest() for name in RUNTIME_FILES}\\n    return hashlib.sha256(canonical_json(commitments)).hexdigest()\\n\\n\\ndef _wheel_observation(root_fd, name):\\n    \\"\\"\\"Hash and inspect one wheel through the same no-follow opened inode.\\"\\"\\"\\n    restored = wheel_installation_filename(name)\\n    distribution_name, version = wheel_filename_identity(restored)\\n    flags = os.O_RDONLY | os.O_NOFOLLOW | os.O_NONBLOCK\\n    fd = os.open(name, flags, dir_fd=root_fd)\\n    with os.fdopen(fd, \\"rb\\") as stream:\\n        before = os.fstat(stream.fileno())\\n        if not stat.S_ISREG(before.st_mode) or before.st_size > MAX_WHEEL_BYTES:\\n            raise PolicyError(\\"WHEEL_LIMIT\\")\\n        digest, size = hashlib.sha256(), 0\\n        while chunk := stream.read(1024 * 1024):\\n            size += len(chunk)\\n            if size > MAX_WHEEL_BYTES:\\n                raise PolicyError(\\"WHEEL_LIMIT\\")\\n            digest.update(chunk)\\n        stream.seek(0)\\n        with zipfile.ZipFile(stream) as wheel:\\n            members = wheel.infolist()\\n            for directory in {item.filename.split(\\"/\\", 1)[0] for item in members\\n                              if item.filename.split(\\"/\\", 1)[0].endswith(\\".dist-info\\")}:\\n                validate_wheel_dist_info(directory, distribution_name, version)\\n            candidates = [item for item in members\\n                          if item.filename.count(\\"/\\") == 1 and item.filename.endswith(\\".dist-info/METADATA\\")]\\n            if len(candidates) > 1:\\n                raise PolicyError(\\"WHEEL_METADATA\\")\\n            if candidates:\\n                validate_wheel_dist_info(candidates[0].filename.split(\\"/\\", 1)[0],\\n                                         distribution_name, version)\\n                if candidates[0].file_size > 1024 * 1024:\\n                    raise PolicyError(\\"WHEEL_METADATA\\")\\n                metadata = parse_metadata(wheel.read(candidates[0]), expected_name=distribution_name,\\n                                          expected_version=version)[\\"projection\\"]\\n            else:\\n                metadata = None\\n        after = os.fstat(stream.fileno())\\n        if (before.st_size, before.st_mtime_ns, before.st_ctime_ns) != (after.st_size, after.st_mtime_ns, after.st_ctime_ns):\\n            raise PolicyError(\\"WHEEL_CHANGED\\")\\n    return {\\"dataset_filename\\": name, \\"installation_filename\\": restored, \\"size\\": size,\\n            \\"sha256\\": digest.hexdigest(), \\"metadata\\": metadata}\\n\\n\\ndef _stream_observation(stream):\\n    stream.seek(0)\\n    digest, size = hashlib.sha256(), 0\\n    while chunk := stream.read(1024 * 1024):\\n        size += len(chunk)\\n        digest.update(chunk)\\n    return size, digest.hexdigest()\\n\\n\\ndef _bootstrap(config, wheels, runner):\\n    if not config.execute_bootstrap:\\n        return {\\"recipe_id\\": \\"NOT_RUN\\", \\"wheel_references\\": [], \\"argv\\": [], \\"executed\\": False,\\n                \\"return_code\\": None, \\"stdout_byte_count\\": 0, \\"stderr_byte_count\\": 0,\\n                \\"stdout_sha256\\": None, \\"stderr_sha256\\": None, \\"diagnostic\\": \\"NOT_RUN\\"}\\n    with tempfile.TemporaryDirectory(prefix=\\"h23-v4-wheelhouse-\\") as temporary:\\n        wheel_directory = Path(temporary)\\n        for wheel in wheels:\\n            (wheel_directory / wheel[\\"installation_filename\\"]).symlink_to(\\n                config.wheelhouse / wheel[\\"dataset_filename\\"]\\n            )\\n        ordered = sorted(wheels, key=lambda wheel: wheel[\\"installation_filename\\"])\\n        argv = [str(config.executable), \\"-m\\", \\"pip\\", \\"install\\", \\"-q\\", \\"--no-deps\\", \\"--force-reinstall\\",\\n                *(str(wheel_directory / wheel[\\"installation_filename\\"]) for wheel in ordered)]\\n        with tempfile.TemporaryFile() as stdout, tempfile.TemporaryFile() as stderr:\\n            try:\\n                completed = runner(argv, stdout=stdout, stderr=stderr, check=False,\\n                                   env={**os.environ, **OFFICIAL_ENVIRONMENT})\\n                return_code = completed.returncode\\n                diagnostic = \\"EXIT_ZERO\\" if return_code == 0 else \\"EXIT_NONZERO\\"\\n            except OSError:\\n                return_code, diagnostic = None, \\"EXECUTION_ERROR\\"\\n            stdout_size, stdout_hash = _stream_observation(stdout)\\n            stderr_size, stderr_hash = _stream_observation(stderr)\\n    return {\\"recipe_id\\": \\"OFFICIAL_NOTEBOOK_CELL_2\\",\\n            \\"wheel_references\\": [wheel[\\"dataset_filename\\"] for wheel in ordered], \\"argv\\": argv,\\n            \\"executed\\": True, \\"return_code\\": return_code,\\n            \\"stdout_byte_count\\": stdout_size, \\"stderr_byte_count\\": stderr_size,\\n            \\"stdout_sha256\\": stdout_hash, \\"stderr_sha256\\": stderr_hash, \\"diagnostic\\": diagnostic}\\n\\n\\ndef _optional_read(root_fd, path, max_bytes=MAX_PAYLOAD_BYTES):\\n    directory, basename = path.split(\\"/\\")\\n    parent = os.open(directory, os.O_RDONLY | os.O_DIRECTORY | os.O_NOFOLLOW, dir_fd=root_fd)\\n    try:\\n        os.stat(basename, dir_fd=parent, follow_symlinks=False)\\n    except FileNotFoundError:\\n        return None\\n    finally:\\n        os.close(parent)\\n    return read_regular(root_fd, path, max_bytes=max_bytes)\\n\\n\\ndef _retain_blob(blobs, observed):\\n    if observed.data is None or observed.size > MAX_PAYLOAD_BYTES:\\n        raise PolicyError(\\"PAYLOAD_LIMIT\\")\\n    if observed.sha256 not in blobs:\\n        if len(blobs) >= MAX_PAYLOADS or sum(map(len, blobs.values())) + observed.size > MAX_EXPANDED_BYTES:\\n            raise PolicyError(\\"PAYLOAD_LIMIT\\")\\n        blobs[observed.sha256] = observed.data\\n\\n\\ndef _distribution_observations(config, interpreter, blobs):\\n    distributions, records, observations, seen = [], [], [], set()\\n    with anchor_directory(config.scripts_directory) as scripts_fd:\\n        for root in config.site_roots:\\n            with anchor_directory(root) as root_fd:\\n                directories = os.listdir(root_fd)\\n                if len(directories) > 8192:\\n                    raise PolicyError(\\"DIRECTORY_LIMIT\\")\\n                for directory in sorted(directories):\\n                    if not directory.endswith(\\".dist-info\\"):\\n                        continue\\n                    matched = [name for name in TARGETS if directory.startswith(name.replace(\\"-\\", \\"_\\") + \\"-\\")]\\n                    if not matched:\\n                        continue\\n                    name = matched[0]\\n                    directory_version = directory[len(name.replace(\\"-\\", \\"_\\")) + 1:-10]\\n                    if name in seen:\\n                        raise PolicyError(\\"DISTRIBUTION_DUPLICATE\\")\\n                    seen.add(name)\\n                    metadata_file = read_regular(root_fd, f\\"{directory}/METADATA\\", max_bytes=1024 * 1024)\\n                    metadata = parse_metadata(metadata_file.data, expected_name=name, expected_version=directory_version)\\n                    direct = _optional_read(root_fd, f\\"{directory}/direct_url.json\\", 512 * 1024)\\n                    entry_points = _optional_read(root_fd, f\\"{directory}/entry_points.txt\\", 512 * 1024)\\n                    distribution = {\\n                        \\"name\\": name, \\"version\\": metadata[\\"projection\\"][\\"Version\\"],\\n                        \\"installation_root\\": str(root), \\"dist_info\\": directory, \\"metadata\\": metadata,\\n                        \\"direct_url\\": {\\"present\\": direct is not None, \\"sha256\\": direct.sha256 if direct else None,\\n                                       \\"size\\": direct.size if direct else None, \\"type\\": \\"REGULAR\\" if direct else \\"ABSENT\\"},\\n                        \\"console_scripts\\": parse_console_scripts(entry_points.data) if entry_points else [],\\n                    }\\n                    record = read_regular(root_fd, f\\"{directory}/RECORD\\", max_bytes=MAX_PAYLOAD_BYTES)\\n                    classified = classify_record_rows(distribution, interpreter, record.data)\\n                    records.append({\\"distribution\\": name, \\"blob\\": record.sha256})\\n                    _retain_blob(blobs, record)\\n                    for row, category in classified:\\n                        include = payload_required(distribution, row.path, category)\\n                        if category == \\"verified_console_script\\":\\n                            observed = read_regular(scripts_fd, row.path.rsplit(\\"/\\", 1)[-1],\\n                                                    max_bytes=MAX_PAYLOAD_BYTES, reject_hardlinks=True)\\n                        elif row.path == f\\"{directory}/RECORD\\":\\n                            observed = record\\n                        elif row.path == f\\"{directory}/METADATA\\":\\n                            observed = metadata_file\\n                        elif row.path == f\\"{directory}/direct_url.json\\" and direct is not None:\\n                            observed = direct\\n                        elif row.path == f\\"{directory}/entry_points.txt\\" and entry_points is not None:\\n                            observed = entry_points\\n                        else:\\n                            observed = read_regular(root_fd, row.path,\\n                                                    max_bytes=MAX_PAYLOAD_BYTES if include else 2 * 2**30,\\n                                                    include=include)\\n                        payload = observed.sha256 if include else None\\n                        if include:\\n                            _retain_blob(blobs, observed)\\n                        observations.append({\\"distribution\\": name, \\"path\\": row.path, \\"kind\\": \\"REGULAR\\",\\n                                             \\"size\\": observed.size, \\"sha256\\": observed.sha256,\\n                                             \\"links\\": observed.nlink, \\"payload\\": payload})\\n                        if len(observations) > 100_000:\\n                            raise PolicyError(\\"OBSERVATION_LIMIT\\")\\n                    distributions.append(distribution)\\n    return distributions, records, observations\\n\\n\\ndef run_capture(config: CaptureConfig, *, pip_runner=None):\\n    \\"\\"\\"Return a new archive path; observations remain untrusted even after import.\\"\\"\\"\\n    interpreter = {\\"python_version\\": list(config.python_version), \\"scheme\\": config.scheme,\\n                   \\"prefix\\": str(config.prefix), \\"site_packages\\": [str(root) for root in config.site_roots],\\n                   \\"scripts_directory\\": str(config.scripts_directory), \\"executable\\": str(config.executable)}\\n    validate_interpreter_roots(interpreter)\\n    wheels = []\\n    with anchor_directory(config.wheelhouse) as wheel_fd:\\n        names = os.listdir(wheel_fd)\\n        if len(names) > 8192:\\n            raise PolicyError(\\"DIRECTORY_LIMIT\\")\\n        for name in sorted(names):\\n            if name.endswith(\\".whl\\") and \\"cutlass\\" not in name.lower():\\n                wheels.append(_wheel_observation(wheel_fd, name))\\n                if len(wheels) > 256:\\n                    raise PolicyError(\\"WHEEL_LIMIT\\")\\n    if len({wheel[\\"installation_filename\\"] for wheel in wheels}) != len(wheels):\\n        raise PolicyError(\\"WHEEL_FILENAME_COLLISION\\")\\n    bootstrap = _bootstrap(config, wheels, pip_runner or subprocess.run)\\n    blobs = {}\\n    distributions, records, observations = _distribution_observations(config, interpreter, blobs)\\n    notebook_hash = hashlib.sha256(config.executed_notebook.read_bytes()).hexdigest() if config.executed_notebook else None\\n    utc = dt.datetime.now(dt.timezone.utc).strftime(\\"%Y%m%dT%H%M%SZ\\")\\n    manifest = {\\n        \\"protocol\\": \\"H23_CAPTURE_PROTOCOL_V4\\", \\"captured_utc\\": utc,\\n        \\"capture_program\\": {\\"sha256\\": capture_program_sha256(), \\"notebook_sha256\\": notebook_hash},\\n        \\"interpreter\\": interpreter, \\"bootstrap\\": bootstrap, \\"wheels\\": wheels,\\n        \\"distributions\\": distributions, \\"record_evidence\\": records, \\"file_observations\\": observations,\\n        \\"payload_references\\": sorted(blobs),\\n    }\\n    validate_manifest(manifest)\\n    archive_bytes = build_archive(manifest, blobs)\\n    # Fixed member names and content addressing; no remote source-shaped tree.\\n    with anchor_directory(config.output_directory) as output_fd:\\n        name = f\\"h23_capture_v4_{utc}.zip\\"\\n        archive_fd = os.open(name, os.O_WRONLY | os.O_CREAT | os.O_EXCL | os.O_NOFOLLOW, 0o600, dir_fd=output_fd)\\n        try:\\n            with os.fdopen(archive_fd, \\"wb\\") as stream:\\n                stream.write(archive_bytes)\\n        except BaseException:\\n            os.unlink(name, dir_fd=output_fd)\\n            raise\\n    return config.output_directory / name\\n", "tools/h23_v4/filesystem.py": "\\"\\"\\"Descriptor-relative reads and a single-file, no-replace evidence transaction.\\n\\nThe configured private evidence root MUST already exist. A committed evidence\\nZIP contains the exact capture.zip snapshot and receipt.json. A hard-link into\\nthe committed namespace atomically publishes both, without replacement.\\n\\"\\"\\"\\n\\nfrom contextlib import contextmanager\\nfrom dataclasses import dataclass\\nimport fcntl\\nimport hashlib\\nimport io\\nimport os\\nfrom pathlib import Path\\nimport re\\nimport stat\\nimport uuid\\nimport zipfile\\n\\nfrom .schema import PolicyError\\n\\nMAX_SNAPSHOT_BYTES = 128 * 1024 * 1024\\nMAX_RECEIPT_BYTES = 4 * 1024 * 1024\\nMAX_FILE_BYTES = 32 * 1024 * 1024\\nMAX_STALE_STAGING = 16\\nMAX_ROOT_ENTRIES = 8192\\n_STAGING = re.compile(r\\"\\\\.h23v4-staging-[0-9a-f]{32}\\\\Z\\")\\n_STAGED_FILES = frozenset((\\"bundle.tmp\\", \\"bundle.prepared\\"))\\n_DIR_FLAGS = os.O_RDONLY | os.O_DIRECTORY | os.O_NOFOLLOW\\n_FILE_FLAGS = os.O_RDONLY | os.O_NOFOLLOW | os.O_NONBLOCK\\n\\n\\n@dataclass(frozen=True)\\nclass FileSnapshot:\\n    size: int\\n    sha256: str\\n    nlink: int\\n    data: bytes | None\\n\\n\\n@dataclass(frozen=True)\\nclass EvidenceRoot:\\n    fd: int\\n    path: Path\\n    protected_roots: tuple\\n\\n\\n@dataclass(frozen=True)\\nclass Publication:\\n    path: Path\\n    archive_sha256: str\\n    status: str\\n    durability: str\\n    warning_codes: tuple\\n\\n\\ndef _parts(relative):\\n    if type(relative) is not str or not relative or relative.startswith(\\"/\\") or \\"\\\\\\\\\\" in relative:\\n        raise PolicyError(\\"LOCAL_PATH\\")\\n    if re.match(r\\"^[A-Za-z]:\\", relative) or any(ord(char) < 32 or ord(char) == 127 for char in relative):\\n        raise PolicyError(\\"LOCAL_PATH\\")\\n    parts = relative.split(\\"/\\")\\n    if any(part in (\\"\\", \\".\\", \\"..\\") for part in parts):\\n        raise PolicyError(\\"LOCAL_PATH\\")\\n    return parts\\n\\n\\ndef _absolute(path):\\n    try:\\n        raw = os.fspath(path)\\n    except TypeError:\\n        raise PolicyError(\\"LOCAL_ROOT_PATH\\") from None\\n    if type(raw) is not str or not raw.startswith(\\"/\\") or \\"\\\\\\\\\\" in raw:\\n        raise PolicyError(\\"LOCAL_ROOT_PATH\\")\\n    if raw != \\"/\\" and (raw.endswith(\\"/\\") or \\"//\\" in raw):\\n        raise PolicyError(\\"LOCAL_ROOT_PATH\\")\\n    if any(ord(char) < 32 or ord(char) == 127 for char in raw):\\n        raise PolicyError(\\"LOCAL_ROOT_PATH\\")\\n    if any(part in (\\".\\", \\"..\\") for part in raw.split(\\"/\\")[1:]):\\n        raise PolicyError(\\"LOCAL_ROOT_PATH\\")\\n    return Path(raw)\\n\\n\\ndef _private(info):\\n    if not stat.S_ISDIR(info.st_mode) or info.st_uid != os.getuid() or info.st_mode & 0o077:\\n        raise PolicyError(\\"LOCAL_ROOT_PERMISSIONS\\")\\n\\n\\ndef _overlaps(left, right):\\n    return left == right or left in right.parents or right in left.parents\\n\\n\\ndef _protected(path, protected_roots):\\n    try:\\n        path = path.resolve(strict=False)\\n    except (OSError, RuntimeError):\\n        raise PolicyError(\\"LOCAL_ROOT_UNAVAILABLE\\") from None\\n    for protected in protected_roots:\\n        target = Path(protected)\\n        if not target.is_absolute():\\n            raise PolicyError(\\"LOCAL_PROTECTED_ROOT\\")\\n        # Protected paths are trusted local configuration, never archive values.\\n        try:\\n            target = target.resolve(strict=False)\\n        except (OSError, RuntimeError):\\n            raise PolicyError(\\"LOCAL_PROTECTED_ROOT\\") from None\\n        if _overlaps(path, target):\\n            raise PolicyError(\\"LOCAL_ROOT_PROTECTED\\")\\n\\n\\ndef _reject_git_directory(descriptor):\\n    \\"\\"\\"Recognize conventional worktree and bare markers without reading Git.\\n\\n    Unusual externally configured Git directories require trusted exclusions.\\n    Marker aliases are never followed and fail closed when the shape matches.\\n    \\"\\"\\"\\n    try:\\n        os.stat(\\".git\\", dir_fd=descriptor, follow_symlinks=False)\\n    except FileNotFoundError:\\n        pass\\n    else:\\n        raise PolicyError(\\"LOCAL_ROOT_GIT\\")\\n    markers = {}\\n    for name in (\\"HEAD\\", \\"objects\\", \\"refs\\"):\\n        try:\\n            markers[name] = os.stat(name, dir_fd=descriptor, follow_symlinks=False)\\n        except FileNotFoundError:\\n            return\\n    head = markers[\\"HEAD\\"].st_mode\\n    directories = (markers[\\"objects\\"].st_mode, markers[\\"refs\\"].st_mode)\\n    if ((stat.S_ISREG(head) or stat.S_ISLNK(head))\\n            and all(stat.S_ISDIR(mode) or stat.S_ISLNK(mode) for mode in directories)):\\n        raise PolicyError(\\"LOCAL_ROOT_GIT\\")\\n\\n\\n@contextmanager\\ndef anchor_directory(path, *, private=False, protected_roots=(), reject_git=False):\\n    \\"\\"\\"Open EVERY absolute-path component from / without following symlinks.\\"\\"\\"\\n    path = _absolute(path)\\n    if not path.is_absolute() or path.anchor != \\"/\\" or any(part in (\\".\\", \\"..\\") for part in path.parts):\\n        raise PolicyError(\\"LOCAL_ROOT_PATH\\")\\n    _protected(path, protected_roots)\\n    descriptor = None\\n    try:\\n        descriptor = os.open(\\"/\\", _DIR_FLAGS)\\n        if reject_git:\\n            _reject_git_directory(descriptor)\\n        for component in path.parts[1:]:\\n            next_fd = os.open(component, _DIR_FLAGS, dir_fd=descriptor)\\n            os.close(descriptor)\\n            descriptor = next_fd\\n            if reject_git:\\n                _reject_git_directory(descriptor)\\n        if private:\\n            _private(os.fstat(descriptor))\\n        # Opening a symlink after a path race fails above; no directory is made.\\n        _protected(path, protected_roots)\\n        yield descriptor\\n    except PolicyError:\\n        raise\\n    except OSError:\\n        raise PolicyError(\\"LOCAL_ROOT_UNAVAILABLE\\") from None\\n    finally:\\n        if descriptor is not None:\\n            os.close(descriptor)\\n\\n\\n@contextmanager\\ndef anchored_root(root, protected_roots):\\n    with anchor_directory(root, private=True, protected_roots=protected_roots, reject_git=True) as descriptor:\\n        handle = EvidenceRoot(descriptor, Path(root), tuple(protected_roots))\\n        _dedicated_root(handle)\\n        yield handle\\n\\n\\ndef read_regular(dir_fd, relative, *, max_bytes=MAX_FILE_BYTES, include=True, reject_hardlinks=False, private=False):\\n    \\"\\"\\"Observe hash and size from one opened stream; optionally retain its bytes.\\"\\"\\"\\n    parts = _parts(relative)\\n    parent = os.dup(dir_fd)\\n    file_fd = None\\n    try:\\n        for component in parts[:-1]:\\n            next_fd = os.open(component, _DIR_FLAGS, dir_fd=parent)\\n            os.close(parent)\\n            parent = next_fd\\n        file_fd = os.open(parts[-1], _FILE_FLAGS, dir_fd=parent)\\n        before = os.fstat(file_fd)\\n        if not stat.S_ISREG(before.st_mode) or (reject_hardlinks and before.st_nlink != 1):\\n            raise PolicyError(\\"LOCAL_FILE_TYPE\\")\\n        if private and (before.st_uid != os.getuid() or stat.S_IMODE(before.st_mode) != 0o600):\\n            raise PolicyError(\\"LOCAL_FILE_PERMISSIONS\\")\\n        if before.st_size > max_bytes:\\n            raise PolicyError(\\"LOCAL_FILE_SIZE\\")\\n        digest = hashlib.sha256()\\n        chunks = [] if include else None\\n        total = 0\\n        while True:\\n            chunk = os.read(file_fd, min(65536, max_bytes - total + 1))\\n            if not chunk:\\n                break\\n            total += len(chunk)\\n            if total > max_bytes:\\n                raise PolicyError(\\"LOCAL_FILE_SIZE\\")\\n            digest.update(chunk)\\n            if include:\\n                chunks.append(chunk)\\n        after = os.fstat(file_fd)\\n        if reject_hardlinks and after.st_nlink != 1:\\n            raise PolicyError(\\"LOCAL_FILE_TYPE\\")\\n        if (before.st_dev, before.st_ino, before.st_size, before.st_mtime_ns, before.st_ctime_ns) != (\\n                after.st_dev, after.st_ino, after.st_size, after.st_mtime_ns, after.st_ctime_ns) or total != before.st_size:\\n            raise PolicyError(\\"LOCAL_FILE_CHANGED\\")\\n        return FileSnapshot(total, digest.hexdigest(), before.st_nlink, b\\"\\".join(chunks) if include else None)\\n    except PolicyError:\\n        raise\\n    except OSError:\\n        raise PolicyError(\\"LOCAL_FILE_UNAVAILABLE\\") from None\\n    finally:\\n        if file_fd is not None:\\n            os.close(file_fd)\\n        os.close(parent)\\n\\n\\ndef load_snapshot(path, max_bytes=MAX_SNAPSHOT_BYTES):\\n    path = Path(path)\\n    if not path.is_absolute():\\n        raise PolicyError(\\"LOCAL_SNAPSHOT_PATH\\")\\n    with anchor_directory(path.parent) as descriptor:\\n        return read_regular(descriptor, path.name, max_bytes=max_bytes).data\\n\\n\\n@contextmanager\\ndef _locked(handle):\\n    descriptor = os.open(\\".h23v4.lock\\", os.O_RDWR | os.O_CREAT | os.O_NOFOLLOW | os.O_NONBLOCK,\\n                         0o600, dir_fd=handle.fd)\\n    try:\\n        info = os.fstat(descriptor)\\n        if not stat.S_ISREG(info.st_mode) or info.st_nlink != 1 or info.st_uid != os.getuid() or info.st_mode & 0o077:\\n            raise PolicyError(\\"LOCAL_LOCK\\")\\n        fcntl.flock(descriptor, fcntl.LOCK_EX)\\n        yield\\n    finally:\\n        os.close(descriptor)\\n\\n\\ndef _reserve_root_capacity(handle, final_name):\\n    \\"\\"\\"Read-only reservation: missing lock plus new stage and final entry.\\n\\n    An existing final entry needs no stage/final slots. It is still independently\\n    checked under the publication lock before an idempotent result is returned.\\n    Remnants are not credited until recovery has actually removed them.\\n    \\"\\"\\"\\n    names = set()\\n    with os.scandir(handle.fd) as iterator:\\n        for entry in iterator:\\n            names.add(entry.name)\\n            if len(names) > MAX_ROOT_ENTRIES:\\n                raise PolicyError(\\"LOCAL_ROOT_LIMIT\\")\\n    required = int(\\".h23v4.lock\\" not in names) + (0 if final_name in names else 2)\\n    if len(names) + required > MAX_ROOT_ENTRIES:\\n        raise PolicyError(\\"LOCAL_ROOT_LIMIT\\")\\n\\n\\ndef _checked_staged_entries(descriptor):\\n    _private(os.fstat(descriptor))\\n    entries = []\\n    with os.scandir(descriptor) as iterator:\\n        for entry in iterator:\\n            entries.append(entry.name)\\n            if len(entries) > len(_STAGED_FILES) or entry.name not in _STAGED_FILES:\\n                raise PolicyError(\\"LOCAL_STAGING\\")\\n    for entry in entries:\\n        info = os.stat(entry, dir_fd=descriptor, follow_symlinks=False)\\n        if not stat.S_ISREG(info.st_mode) or info.st_uid != os.getuid():\\n            raise PolicyError(\\"LOCAL_STAGING\\")\\n    return entries\\n\\n\\ndef _dedicated_root(handle):\\n    \\"\\"\\"Read-only, bounded preflight; no foreign directory trees are permitted.\\n\\n    Together with ancestor marker checks, this excludes unrelated Git worktrees\\n    without recursively scanning arbitrary local directory trees.\\n    \\"\\"\\"\\n    with os.scandir(handle.fd) as iterator:\\n        for count, entry in enumerate(iterator, 1):\\n            if count > MAX_ROOT_ENTRIES:\\n                raise PolicyError(\\"LOCAL_ROOT_LIMIT\\")\\n            if entry.name == \\".git\\":\\n                raise PolicyError(\\"LOCAL_ROOT_GIT\\")\\n            info = entry.stat(follow_symlinks=False)\\n            if stat.S_ISREG(info.st_mode):\\n                continue\\n            if not stat.S_ISDIR(info.st_mode) or _STAGING.fullmatch(entry.name) is None:\\n                raise PolicyError(\\"LOCAL_ROOT_NOT_DEDICATED\\")\\n            descriptor = os.open(entry.name, _DIR_FLAGS, dir_fd=handle.fd)\\n            try:\\n                _checked_staged_entries(descriptor)\\n            finally:\\n                os.close(descriptor)\\n\\n\\ndef _remove_staging(root_fd, name):\\n    \\"\\"\\"Bounded cleanup removes only our exact staged regular-file names.\\"\\"\\"\\n    if _STAGING.fullmatch(name) is None:\\n        raise PolicyError(\\"LOCAL_STAGING\\")\\n    descriptor = os.open(name, _DIR_FLAGS, dir_fd=root_fd)\\n    try:\\n        entries = _checked_staged_entries(descriptor)\\n        for entry in entries:\\n            os.unlink(entry, dir_fd=descriptor)\\n    finally:\\n        os.close(descriptor)\\n    os.rmdir(name, dir_fd=root_fd)\\n\\n\\ndef recover_staging(handle, max_remnants=MAX_STALE_STAGING):\\n    \\"\\"\\"Recover recognized remnants; never touch committed evidence or aliases.\\"\\"\\"\\n    remnants = []\\n    with os.scandir(handle.fd) as iterator:\\n        for count, entry in enumerate(iterator, 1):\\n            if count > MAX_ROOT_ENTRIES:\\n                raise PolicyError(\\"LOCAL_ROOT_LIMIT\\")\\n            if _STAGING.fullmatch(entry.name):\\n                remnants.append(entry.name)\\n                if len(remnants) > max_remnants:\\n                    raise PolicyError(\\"LOCAL_STAGING_LIMIT\\")\\n    for name in sorted(remnants):\\n        _remove_staging(handle.fd, name)\\n\\n\\ndef _bundle(snapshot, receipt):\\n    if type(snapshot) is not bytes or len(snapshot) > MAX_SNAPSHOT_BYTES:\\n        raise PolicyError(\\"LOCAL_SNAPSHOT_SIZE\\")\\n    if type(receipt) is not bytes or len(receipt) > MAX_RECEIPT_BYTES:\\n        raise PolicyError(\\"LOCAL_RECEIPT_SIZE\\")\\n    stream = io.BytesIO()\\n    with zipfile.ZipFile(stream, \\"w\\", compression=zipfile.ZIP_STORED) as archive:\\n        for name, payload in ((\\"capture.zip\\", snapshot), (\\"receipt.json\\", receipt)):\\n            info = zipfile.ZipInfo(name, date_time=(1980, 1, 1, 0, 0, 0))\\n            info.create_system = 3\\n            info.external_attr = (stat.S_IFREG | 0o600) << 16\\n            archive.writestr(info, payload)\\n    return stream.getvalue()\\n\\n\\ndef _write_new(dir_fd, name, data):\\n    descriptor = os.open(name, os.O_WRONLY | os.O_CREAT | os.O_EXCL | os.O_NOFOLLOW,\\n                         0o600, dir_fd=dir_fd)\\n    try:\\n        if not stat.S_ISREG(os.fstat(descriptor).st_mode):\\n            raise PolicyError(\\"LOCAL_FILE_TYPE\\")\\n        view = memoryview(data)\\n        while view:\\n            count = os.write(descriptor, view)\\n            if count <= 0:\\n                raise PolicyError(\\"LOCAL_WRITE\\")\\n            view = view[count:]\\n        os.fsync(descriptor)\\n    finally:\\n        os.close(descriptor)\\n\\n\\ndef _existing_bundle(handle, final_name):\\n    try:\\n        return read_regular(handle.fd, final_name,\\n                            max_bytes=MAX_SNAPSHOT_BYTES + MAX_RECEIPT_BYTES + 4096,\\n                            reject_hardlinks=True, private=True)\\n    except PolicyError as error:\\n        if error.code != \\"LOCAL_FILE_UNAVAILABLE\\":\\n            raise PolicyError(\\"EVIDENCE_CONFLICT\\") from None\\n        # A no-follow read failure is absence only when no entry exists.\\n        try:\\n            os.stat(final_name, dir_fd=handle.fd, follow_symlinks=False)\\n        except FileNotFoundError:\\n            return None\\n        raise PolicyError(\\"EVIDENCE_CONFLICT\\") from None\\n\\n\\ndef _fsync_existing_bundle(handle, final_name):\\n    \\"\\"\\"Reconfirm the already verified existing file and its published name.\\"\\"\\"\\n    descriptor = os.open(final_name, _FILE_FLAGS, dir_fd=handle.fd)\\n    try:\\n        info = os.fstat(descriptor)\\n        if (not stat.S_ISREG(info.st_mode) or info.st_nlink != 1\\n                or info.st_uid != os.getuid() or stat.S_IMODE(info.st_mode) != 0o600):\\n            raise PolicyError(\\"LOCAL_FILE_TYPE\\")\\n        os.fsync(descriptor)\\n    finally:\\n        os.close(descriptor)\\n    os.fsync(handle.fd)\\n\\n\\ndef publish_evidence(handle, snapshot_bytes, receipt_bytes):\\n    \\"\\"\\"Atomically publish a complete bundle, or preserve an exact prior bundle.\\n\\n    The successful no-replace link is the commit point. Subsequent durability\\n    and cleanup errors are bounded warnings and never turn a committed result\\n    into rejection. Already committed bytes are never rolled back or changed.\\n    \\"\\"\\"\\n    _private(os.fstat(handle.fd))\\n    digest = hashlib.sha256(snapshot_bytes).hexdigest()\\n    final_name = digest + \\".evidence.zip\\"\\n    data = _bundle(snapshot_bytes, receipt_bytes)\\n    status = None\\n    warnings = set()\\n    durability = \\"CONFIRMED\\"\\n\\n    def confirm_durability(operation):\\n        nonlocal durability\\n        try:\\n            operation()\\n        except (OSError, PolicyError):\\n            durability = \\"UNCONFIRMED\\"\\n            warnings.add(\\"PUBLICATION_DURABILITY_UNCONFIRMED\\")\\n\\n    try:\\n        # Reserve before even creating the fixed lock entry. New publication\\n        # needs at most three slots; exact reimport needs only a missing lock.\\n        _reserve_root_capacity(handle, final_name)\\n        with _locked(handle):\\n            recover_staging(handle)\\n            existing = _existing_bundle(handle, final_name)\\n            if existing is not None:\\n                if existing.data != data:\\n                    raise PolicyError(\\"EVIDENCE_CONFLICT\\")\\n                status = \\"IDEMPOTENT\\"\\n                confirm_durability(lambda: _fsync_existing_bundle(handle, final_name))\\n            else:\\n                # Recheck under the lock after recovery, before new child entries.\\n                _reserve_root_capacity(handle, final_name)\\n                stage_name = \\".h23v4-staging-\\" + uuid.uuid4().hex\\n                stage_fd = None\\n                created = False\\n                try:\\n                    os.mkdir(stage_name, mode=0o700, dir_fd=handle.fd)\\n                    created = True\\n                    stage_fd = os.open(stage_name, _DIR_FLAGS, dir_fd=handle.fd)\\n                    _private(os.fstat(stage_fd))\\n                    _write_new(stage_fd, \\"bundle.tmp\\", data)\\n                    os.rename(\\"bundle.tmp\\", \\"bundle.prepared\\", src_dir_fd=stage_fd, dst_dir_fd=stage_fd)\\n                    os.fsync(stage_fd)\\n                    try:\\n                        os.link(\\"bundle.prepared\\", final_name, src_dir_fd=stage_fd, dst_dir_fd=handle.fd,\\n                                follow_symlinks=False)\\n                    except FileExistsError:\\n                        existing = _existing_bundle(handle, final_name)\\n                        if existing is None or existing.data != data:\\n                            raise PolicyError(\\"EVIDENCE_CONFLICT\\")\\n                        status = \\"IDEMPOTENT\\"\\n                        confirm_durability(lambda: _fsync_existing_bundle(handle, final_name))\\n                    else:\\n                        status = \\"CREATED\\"\\n                        confirm_durability(lambda: os.fsync(handle.fd))\\n                finally:\\n                    cleanup_error = None\\n                    if stage_fd is not None:\\n                        try:\\n                            os.close(stage_fd)\\n                        except OSError as error:\\n                            cleanup_error = error\\n                    if created:\\n                        try:\\n                            _remove_staging(handle.fd, stage_name)\\n                        except (OSError, PolicyError) as error:\\n                            cleanup_error = error\\n                    if cleanup_error is not None:\\n                        if status is None:\\n                            raise cleanup_error\\n                        warnings.add(\\"PUBLICATION_CLEANUP_UNCONFIRMED\\")\\n                # Confirm cleanup changes too; any prior failure stays explicit.\\n                confirm_durability(lambda: os.fsync(handle.fd))\\n    except (OSError, PolicyError) as error:\\n        if status is None:\\n            if isinstance(error, PolicyError):\\n                raise\\n            raise PolicyError(\\"LOCAL_STORAGE\\") from None\\n        # Descriptor/lock teardown is also after the irrevocable commit point.\\n        warnings.add(\\"PUBLICATION_CLEANUP_UNCONFIRMED\\")\\n    return Publication(handle.path / final_name, digest, status, durability, tuple(sorted(warnings)))\\n", "tools/h23_v4/metadata_policy.py": "\\"\\"\\"Metadata is projected, never redacted or copied as arbitrary free text.\\"\\"\\"\\n\\nimport hashlib\\nimport configparser\\nimport re\\n\\nfrom tools.h23_v4.schema import PolicyError, exact_fields, integer, sha256, string\\n\\nMAX_METADATA_BYTES = 512 * 1024\\nSUPPORTED_METADATA_VERSIONS = frozenset((\\"1.0\\", \\"1.1\\", \\"1.2\\", \\"2.1\\", \\"2.2\\", \\"2.3\\", \\"2.4\\"))\\n_NAME = re.compile(r\\"[A-Za-z0-9](?:[A-Za-z0-9._-]*[A-Za-z0-9])?\\\\Z\\")\\n# Intentionally bounded ASCII packaging profile, including wheel local versions.\\n_VERSION = re.compile(r\\"(?:[0-9]+!)?[0-9]+(?:\\\\.[0-9]+)*(?:(?:a|b|rc)[0-9]+)?(?:\\\\.post[0-9]+)?(?:\\\\.dev[0-9]+)?(?:\\\\+[a-z0-9]+(?:[._-][a-z0-9]+)*)?\\\\Z\\", re.I | re.ASCII)\\n_HEADER = re.compile(r\\"([A-Za-z][A-Za-z0-9-]*):[ \\\\t]*(.*)\\\\Z\\")\\n_PROJECTION_FIELDS = frozenset((\\"metadata-version\\", \\"name\\", \\"version\\"))\\n_SINGLETONS = _PROJECTION_FIELDS | frozenset((\\"requires-python\\", \\"summary\\", \\"description\\", \\"description-content-type\\", \\"keywords\\", \\"home-page\\", \\"download-url\\", \\"author\\", \\"author-email\\", \\"maintainer\\", \\"maintainer-email\\", \\"license\\", \\"license-expression\\"))\\n\\n\\ndef canonical_name(name):\\n    if type(name) is not str or len(name) > 128 or _NAME.fullmatch(name) is None:\\n        raise PolicyError(\\"METADATA_NAME\\")\\n    return re.sub(r\\"[-_.]+\\", \\"-\\", name).lower()\\n\\n\\ndef validate_projection(projection, expected_name=None, expected_version=None):\\n    exact_fields(projection, (\\"Metadata-Version\\", \\"Name\\", \\"Version\\"))\\n    for value in projection.values():\\n        string(value)\\n    if projection[\\"Metadata-Version\\"] not in SUPPORTED_METADATA_VERSIONS:\\n        raise PolicyError(\\"METADATA_VERSION_UNSUPPORTED\\")\\n    name = canonical_name(projection[\\"Name\\"])\\n    if projection[\\"Name\\"] != name:\\n        raise PolicyError(\\"METADATA_NAME_NONCANONICAL\\")\\n    version = projection[\\"Version\\"]\\n    if len(version) > 128 or _VERSION.fullmatch(version) is None:\\n        raise PolicyError(\\"METADATA_PACKAGE_VERSION\\")\\n    if expected_name is not None and name != canonical_name(expected_name):\\n        raise PolicyError(\\"METADATA_IDENTITY\\")\\n    if expected_version is not None and version != expected_version:\\n        raise PolicyError(\\"METADATA_IDENTITY\\")\\n    return projection\\n\\n\\ndef _wheel_parts(filename):\\n    if type(filename) is not str or len(filename) > 4096 or re.fullmatch(r\\"[A-Za-z0-9][A-Za-z0-9_.+-]*\\\\.whl\\", filename) is None:\\n        raise PolicyError(\\"WHEEL_FILENAME\\")\\n    parts = filename[:-4].split(\\"-\\")\\n    if len(parts) not in (5, 6) or re.fullmatch(r\\"[A-Za-z0-9][A-Za-z0-9_.]*\\", parts[0]) is None:\\n        raise PolicyError(\\"WHEEL_FILENAME\\")\\n    if len(parts) == 6 and re.fullmatch(r\\"[0-9][A-Za-z0-9_]*\\", parts[2]) is None:\\n        raise PolicyError(\\"WHEEL_FILENAME\\")\\n    if any(re.fullmatch(r\\"[A-Za-z0-9_]+(?:\\\\.[A-Za-z0-9_]+)*\\", part) is None for part in parts[-3:]):\\n        raise PolicyError(\\"WHEEL_FILENAME\\")\\n    return parts\\n\\n\\ndef _wheel_version(version):\\n    # The official wheelhouse strips \'+\' before the fixed cu128 local suffix.\\n    if \\"+\\" not in version and version.endswith(\\"cu128\\"):\\n        version = version[:-5] + \\"+cu128\\"\\n    if len(version) > 128 or _VERSION.fullmatch(version) is None:\\n        raise PolicyError(\\"WHEEL_VERSION\\")\\n    return version\\n\\n\\ndef wheel_filename_identity(filename):\\n    \\"\\"\\"Return bounded filename identity, including the supported cu128 alias.\\"\\"\\"\\n    parts = _wheel_parts(filename)\\n    return canonical_name(parts[0]), _wheel_version(parts[1])\\n\\n\\ndef wheel_installation_filename(dataset_filename):\\n    \\"\\"\\"Apply the sole supported wheelhouse rename and validate its identity.\\"\\"\\"\\n    parts = _wheel_parts(dataset_filename)\\n    wheel_filename_identity(dataset_filename)\\n    parts[1] = _wheel_version(parts[1])\\n    return \\"-\\".join(parts) + \\".whl\\"\\n\\n\\ndef validate_wheel_observation_identity(dataset_filename, installation_filename, projection):\\n    \\"\\"\\"Check wheel commitments internally, without an installed-package claim.\\"\\"\\"\\n    expected_filename = wheel_installation_filename(dataset_filename)\\n    name, version = wheel_filename_identity(installation_filename)\\n    if installation_filename != expected_filename:\\n        raise PolicyError(\\"WHEEL_RENAME\\")\\n    if projection is not None:\\n        validate_projection(projection, name, version)\\n    return name, version\\n\\n\\ndef validate_wheel_dist_info(directory, expected_name, expected_version):\\n    \\"\\"\\"Compare internal and filename identity using the same name normalization.\\"\\"\\"\\n    if type(directory) is not str or not directory.endswith(\\".dist-info\\"):\\n        raise PolicyError(\\"WHEEL_DIST_INFO_IDENTITY\\")\\n    parts = directory[:-10].rsplit(\\"-\\", 1)\\n    if len(parts) != 2 or len(parts[1]) > 128 or _VERSION.fullmatch(parts[1]) is None:\\n        raise PolicyError(\\"WHEEL_DIST_INFO_IDENTITY\\")\\n    try:\\n        name = canonical_name(parts[0])\\n    except PolicyError:\\n        raise PolicyError(\\"WHEEL_DIST_INFO_IDENTITY\\") from None\\n    if name != canonical_name(expected_name) or parts[1] != expected_version:\\n        raise PolicyError(\\"WHEEL_DIST_INFO_IDENTITY\\")\\n    return directory\\n\\n\\ndef validate_metadata_observation(observation, expected_name=None, expected_version=None):\\n    exact_fields(observation, (\\"raw_sha256\\", \\"raw_size\\", \\"projection\\"))\\n    sha256(observation[\\"raw_sha256\\"])\\n    integer(observation[\\"raw_size\\"], 1, MAX_METADATA_BYTES)\\n    validate_projection(observation[\\"projection\\"], expected_name, expected_version)\\n    return observation\\n\\n\\ndef parse_metadata(raw, expected_name=None, expected_version=None):\\n    if type(raw) is not bytes or not raw or len(raw) > MAX_METADATA_BYTES:\\n        raise PolicyError(\\"METADATA_LIMIT\\")\\n    try:\\n        text = raw.decode(\\"utf-8\\")\\n    except UnicodeDecodeError:\\n        raise PolicyError(\\"METADATA_SYNTAX\\") from None\\n    if \\"\\\\x00\\" in text or \\"\\\\r\\" in text.replace(\\"\\\\r\\\\n\\", \\"\\"):\\n        raise PolicyError(\\"METADATA_SYNTAX\\")\\n    values = {}\\n    seen_singletons = set()\\n    previous = None\\n    for line in text.replace(\\"\\\\r\\\\n\\", \\"\\\\n\\").split(\\"\\\\n\\"):\\n        if line == \\"\\":\\n            break  # Description body is not metadata projection evidence.\\n        if len(line) > 8192 or any(ord(c) < 32 and c != \\"\\\\t\\" for c in line):\\n            raise PolicyError(\\"METADATA_SYNTAX\\")\\n        if line[:1] in (\\" \\", \\"\\\\t\\"):\\n            if previous is None or previous in _PROJECTION_FIELDS:\\n                raise PolicyError(\\"METADATA_SYNTAX\\")\\n            continue  # Valid continuation for an omitted field.\\n        match = _HEADER.fullmatch(line)\\n        if match is None:\\n            raise PolicyError(\\"METADATA_SYNTAX\\")\\n        key, value = match.groups()\\n        previous = key.lower()\\n        if previous in _SINGLETONS:\\n            if previous in seen_singletons:\\n                raise PolicyError(\\"METADATA_DUPLICATE\\")\\n            seen_singletons.add(previous)\\n        if previous in _PROJECTION_FIELDS:\\n            values[previous] = value\\n    if not {\\"metadata-version\\", \\"name\\", \\"version\\"} <= values.keys():\\n        raise PolicyError(\\"METADATA_REQUIRED\\")\\n    projection = {\\"Metadata-Version\\": values[\\"metadata-version\\"],\\n                  \\"Name\\": canonical_name(values[\\"name\\"]), \\"Version\\": values[\\"version\\"]}\\n    validate_projection(projection, expected_name, expected_version)\\n    return {\\"raw_sha256\\": hashlib.sha256(raw).hexdigest(),\\n            \\"raw_size\\": len(raw), \\"projection\\": projection}\\n\\n\\ndef validate_metadata(raw, expected_name=None, expected_version=None):\\n    return parse_metadata(raw, expected_name, expected_version)[\\"projection\\"]\\n\\n\\ndef validate_console_scripts(declarations):\\n    if not isinstance(declarations, list) or len(declarations) > 128:\\n        raise PolicyError(\\"CONSOLE_DECLARATION_LIMIT\\")\\n    seen = set()\\n    for declaration in declarations:\\n        exact_fields(declaration, (\\"name\\", \\"target\\"))\\n        name, target = declaration[\\"name\\"], declaration[\\"target\\"]\\n        if type(name) is not str or len(name) > 128 or re.fullmatch(r\\"[A-Za-z0-9_][A-Za-z0-9_.-]*\\", name) is None:\\n            raise PolicyError(\\"CONSOLE_DECLARATION\\")\\n        if name.casefold() in seen:\\n            raise PolicyError(\\"CONSOLE_DECLARATION_DUPLICATE\\")\\n        seen.add(name.casefold())\\n        if type(target) is not str or len(target) > 256 or re.fullmatch(r\\"[A-Za-z_][A-Za-z0-9_]*(?:\\\\.[A-Za-z_][A-Za-z0-9_]*)*:[A-Za-z_][A-Za-z0-9_]*(?:\\\\.[A-Za-z_][A-Za-z0-9_]*)*\\", target) is None:\\n            raise PolicyError(\\"CONSOLE_TARGET\\")\\n    return declarations\\n\\n\\ndef parse_console_scripts(raw):\\n    if type(raw) is not bytes or len(raw) > MAX_METADATA_BYTES:\\n        raise PolicyError(\\"ENTRY_POINTS_LIMIT\\")\\n    try:\\n        text = raw.decode(\\"utf-8\\")\\n        if \\"\\\\x00\\" in text:\\n            raise PolicyError(\\"ENTRY_POINTS_SYNTAX\\")\\n        parser = configparser.ConfigParser(interpolation=None, strict=True, delimiters=(\\"=\\",), empty_lines_in_values=False)\\n        parser.optionxform = str\\n        parser.read_string(text)\\n        if parser.defaults():\\n            raise PolicyError(\\"ENTRY_POINTS_SYNTAX\\")\\n        if parser.has_section(\\"gui_scripts\\") and parser.items(\\"gui_scripts\\"):\\n            raise PolicyError(\\"GUI_SCRIPTS_UNSUPPORTED\\")\\n        declarations = [{\\"name\\": name, \\"target\\": target.strip()} for name, target in parser.items(\\"console_scripts\\")] if parser.has_section(\\"console_scripts\\") else []\\n    except (UnicodeDecodeError, configparser.Error):\\n        raise PolicyError(\\"ENTRY_POINTS_SYNTAX\\") from None\\n    validate_console_scripts(declarations)\\n    return sorted(declarations, key=lambda entry: entry[\\"name\\"])\\n", "tools/h23_v4/record_policy.py": "\\"\\"\\"Bounded RECORD policy; categories are derived after safety and ownership.\\"\\"\\"\\n\\nimport base64\\nimport csv\\nfrom dataclasses import dataclass\\nimport hashlib\\nimport io\\nimport posixpath\\nimport re\\n\\nfrom tools.h23_v4.metadata_policy import validate_console_scripts\\nfrom tools.h23_v4.schema import PolicyError\\n\\nMAX_RECORD_BYTES = 20 * 1024 * 1024\\nMAX_RECORD_ROWS = 100_000\\nMAX_RECORD_FIELD = 2048\\nMAX_OBSERVED_SIZE = 2 * 2**30\\nOWNED_PREFIXES = {\\n    \\"swegemma\\": (\\"swegemma\\",), \\"adk-submission\\": (\\"adk_submission\\",),\\n    \\"adk-eval-core\\": (\\"adk_eval_core\\",), \\"google-adk\\": (\\"google\\", \\"adk\\"),\\n    \\"google-genai\\": (\\"google\\", \\"genai\\"), \\"litellm\\": (\\"litellm\\",),\\n    \\"vllm\\": (\\"vllm\\",), \\"transformers\\": (\\"transformers\\",),\\n}\\nHARNESS_DISTRIBUTIONS = frozenset((\\"swegemma\\", \\"adk-submission\\", \\"adk-eval-core\\"))\\nSOURCE_SUFFIXES = frozenset((\\".py\\", \\".pyi\\", \\".json\\", \\".yaml\\", \\".yml\\", \\".toml\\", \\".txt\\", \\".md\\", \\".cfg\\", \\".ini\\", \\".typed\\", \\".j2\\", \\".jinja\\"))\\n_COMPONENT = re.compile(r\\"[A-Za-z0-9_.][A-Za-z0-9_.+-]*\\\\Z\\")\\n_HASH = re.compile(r\\"sha256=([A-Za-z0-9_-]{43})\\\\Z\\")\\n_DECIMAL = re.compile(r\\"(?:0|[1-9][0-9]{0,9})\\\\Z\\")\\n\\n\\n@dataclass(frozen=True)\\nclass RecordRow:\\n    path: str\\n    hash: str\\n    size: str\\n\\n\\ndef parse_record(raw):\\n    if type(raw) is not bytes or not raw or len(raw) > MAX_RECORD_BYTES:\\n        raise PolicyError(\\"RECORD_LIMIT\\")\\n    try:\\n        text = raw.decode(\\"utf-8\\")\\n    except UnicodeDecodeError:\\n        raise PolicyError(\\"RECORD_UTF8\\") from None\\n    if \\"\\\\x00\\" in text:\\n        raise PolicyError(\\"RECORD_NUL\\")\\n    _validate_csv_quoting(text)\\n    rows = []\\n    try:\\n        reader = csv.reader(io.StringIO(text, newline=\\"\\"), delimiter=\\",\\", quotechar=\'\\"\', strict=True)\\n        for fields in reader:\\n            if len(fields) != 3:\\n                raise PolicyError(\\"RECORD_COLUMNS\\")\\n            if any(len(field) > MAX_RECORD_FIELD for field in fields):\\n                raise PolicyError(\\"RECORD_FIELD_LIMIT\\")\\n            rows.append(RecordRow(*fields))\\n            if len(rows) > MAX_RECORD_ROWS:\\n                raise PolicyError(\\"RECORD_ROW_LIMIT\\")\\n    except csv.Error:\\n        raise PolicyError(\\"RECORD_CSV\\") from None\\n    if not rows:\\n        raise PolicyError(\\"RECORD_COLUMNS\\")\\n    return rows\\n\\n\\ndef _validate_csv_quoting(text):\\n    # Python\'s strict reader still accepts quote characters in unquoted fields.\\n    state = \\"start\\"\\n    for index, char in enumerate(text):\\n        if char == \\"\\\\r\\" and (index + 1 == len(text) or text[index + 1] != \\"\\\\n\\"):\\n            raise PolicyError(\\"RECORD_CSV\\")\\n        if state == \\"quoted\\":\\n            if char == \'\\"\':\\n                state = \\"after_quote\\"\\n        elif state == \\"after_quote\\":\\n            if char == \'\\"\':\\n                state = \\"quoted\\"\\n            elif char == \\",\\" or char in \\"\\\\r\\\\n\\":\\n                state = \\"start\\"\\n            else:\\n                raise PolicyError(\\"RECORD_CSV\\")\\n        elif char == \'\\"\':\\n            if state != \\"start\\":\\n                raise PolicyError(\\"RECORD_CSV\\")\\n            state = \\"quoted\\"\\n        elif char == \\",\\" or char in \\"\\\\r\\\\n\\":\\n            state = \\"start\\"\\n        else:\\n            state = \\"unquoted\\"\\n    if state == \\"quoted\\":\\n        raise PolicyError(\\"RECORD_CSV\\")\\n\\n\\ndef canonical_components(path):\\n    if type(path) is not str or not path or len(path) > MAX_RECORD_FIELD or path.startswith(\\"/\\") or \\"\\\\\\\\\\" in path or any(ord(c) < 32 or ord(c) == 127 for c in path):\\n        raise PolicyError(\\"RECORD_PATH\\")\\n    parts = tuple(path.split(\\"/\\"))\\n    if any(part in (\\"\\", \\".\\", \\"..\\") or _COMPONENT.fullmatch(part) is None for part in parts):\\n        raise PolicyError(\\"RECORD_PATH\\")\\n    return parts\\n\\n\\ndef decode_record_hash(value):\\n    match = _HASH.fullmatch(value)\\n    if match is None:\\n        raise PolicyError(\\"RECORD_HASH\\")\\n    try:\\n        decoded = base64.b64decode(match[1] + \\"=\\", altchars=b\\"-_\\", validate=True)\\n    except ValueError:\\n        raise PolicyError(\\"RECORD_HASH\\") from None\\n    if len(decoded) != 32 or base64.urlsafe_b64encode(decoded).decode().rstrip(\\"=\\") != match[1]:\\n        raise PolicyError(\\"RECORD_HASH\\")\\n    return decoded.hex()\\n\\n\\ndef decode_record_size(value):\\n    if _DECIMAL.fullmatch(value) is None:\\n        raise PolicyError(\\"RECORD_SIZE\\")\\n    size = int(value)\\n    if size > MAX_OBSERVED_SIZE:\\n        raise PolicyError(\\"RECORD_SIZE\\")\\n    return size\\n\\n\\ndef validate_interpreter_roots(interpreter):\\n    \\"\\"\\"Each scheme\'s own sysconfig templates, anchored at the observed prefix.\\n\\n    posix_local is Debian\'s default scheme: purelib/platlib are\\n    {base}/local/lib/python{X.Y}/dist-packages and scripts are {base}/local/bin,\\n    while the interpreter stays in {base}/bin or is invoked through that scripts\\n    directory. No absolute path is trusted; a neighbouring root never matches.\\n    \\"\\"\\"\\n    prefix = interpreter[\\"prefix\\"]\\n    major, minor, _micro = interpreter[\\"python_version\\"]\\n    version = f\\"python{major}.{minor}\\"\\n    if interpreter[\\"scheme\\"] in (\\"posix_prefix\\", \\"posix_venv\\"):\\n        expected = {f\\"{prefix}/{lib}/{version}/{suffix}\\" for lib in (\\"lib\\", \\"lib64\\") for suffix in (\\"site-packages\\", \\"dist-packages\\")}\\n        scripts = prefix + \\"/bin\\"\\n        executables = {scripts}\\n    elif interpreter[\\"scheme\\"] == \\"posix_local\\":\\n        expected = {f\\"{prefix}/local/lib/{version}/dist-packages\\"}\\n        scripts = prefix + \\"/local/bin\\"\\n        executables = {prefix + \\"/bin\\", scripts}\\n    else:\\n        raise PolicyError(\\"INTERPRETER_SCHEME\\")\\n    if interpreter[\\"scripts_directory\\"] != scripts or not interpreter[\\"site_packages\\"] or not set(interpreter[\\"site_packages\\"]) <= expected:\\n        raise PolicyError(\\"INTERPRETER_SCHEME\\")\\n    if posixpath.dirname(interpreter[\\"executable\\"]) not in executables:\\n        raise PolicyError(\\"INTERPRETER_SCHEME\\")\\n\\n\\ndef _owned(parts, distribution):\\n    prefix = OWNED_PREFIXES[distribution[\\"name\\"]]\\n    return parts[:len(prefix)] == prefix and len(parts) > len(prefix) or parts[0] == distribution[\\"dist_info\\"] and len(parts) > 1\\n\\n\\ndef console_script_path(distribution, interpreter, basename):\\n    validate_interpreter_roots(interpreter)\\n    relative = posixpath.relpath(interpreter[\\"scripts_directory\\"] + \\"/\\" + basename, distribution[\\"installation_root\\"])\\n    if re.fullmatch(r\\"(?:\\\\.\\\\./)+bin/[A-Za-z0-9_][A-Za-z0-9_.-]*\\", relative) is None:\\n        raise PolicyError(\\"CONSOLE_PATH\\")\\n    return relative\\n\\n\\ndef _check_collision(paths):\\n    folded = {path.casefold() for path in paths}\\n    if len(folded) != len(paths):\\n        raise PolicyError(\\"RECORD_COLLISION\\")\\n    for path in folded:\\n        parts = path.split(\\"/\\")\\n        if any(\\"/\\".join(parts[:index]) in folded for index in range(1, len(parts))):\\n            raise PolicyError(\\"RECORD_COLLISION\\")\\n\\n\\ndef classify_record_rows(distribution, interpreter, record_bytes):\\n    \\"\\"\\"Classify without reading any RECORD-supplied filesystem path.\\"\\"\\"\\n    validate_interpreter_roots(interpreter)\\n    name = distribution[\\"name\\"]\\n    if name not in OWNED_PREFIXES or distribution[\\"installation_root\\"] not in interpreter[\\"site_packages\\"] or distribution[\\"dist_info\\"] != name.replace(\\"-\\", \\"_\\") + \\"-\\" + distribution[\\"version\\"] + \\".dist-info\\":\\n        raise PolicyError(\\"DISTRIBUTION_ROOT\\")\\n    validate_console_scripts(distribution[\\"console_scripts\\"])\\n    declarations = {item[\\"name\\"]: item[\\"target\\"] for item in distribution[\\"console_scripts\\"]}\\n    rows = parse_record(record_bytes)\\n    paths = [row.path for row in rows]\\n    path_set = set(paths)\\n    _check_collision(paths)\\n    own_record = distribution[\\"dist_info\\"] + \\"/RECORD\\"\\n    own_metadata = distribution[\\"dist_info\\"] + \\"/METADATA\\"\\n    if paths.count(own_record) != 1 or paths.count(own_metadata) != 1:\\n        raise PolicyError(\\"RECORD_REQUIRED\\")\\n    classified = []\\n    script_names = set()\\n    for row in rows:\\n        if row.path.startswith(\\"../\\"):\\n            basename = row.path.rsplit(\\"/\\", 1)[-1]\\n            if basename not in declarations or row.path != console_script_path(distribution, interpreter, basename):\\n                raise PolicyError(\\"CONSOLE_PATH\\")\\n            script_names.add(basename)\\n            category = \\"verified_console_script\\"\\n        else:\\n            parts = canonical_components(row.path)\\n            if not _owned(parts, distribution):\\n                raise PolicyError(\\"RECORD_OWNERSHIP\\")\\n            category = \\"owned_regular\\"\\n            if row.path == own_record:\\n                if row.hash != \\"\\" or row.size != \\"\\":\\n                    raise PolicyError(\\"RECORD_SELF\\")\\n                category = \\"excluded_volatile\\"\\n            elif len(parts) == 2 and parts[0] == distribution[\\"dist_info\\"] and parts[1] in (\\"INSTALLER\\", \\"REQUESTED\\", \\"direct_url.json\\"):\\n                category = \\"excluded_volatile\\"\\n            elif \\"__pycache__\\" in parts or row.path.endswith((\\".pyc\\", \\".pyo\\")):\\n                major, minor, _micro = interpreter[\\"python_version\\"]\\n                cache = re.fullmatch(r\\"([A-Za-z_][A-Za-z0-9_]*)\\\\.cpython-\\" + str(major) + str(minor) + r\\"(?:\\\\.opt-[12])?\\\\.pyc\\", parts[-1])\\n                source = \\"/\\".join(parts[:-2] + ((cache[1] + \\".py\\",) if cache else ()))\\n                package_prefix = OWNED_PREFIXES[name]\\n                if len(parts) < len(package_prefix) + 2 or parts[-2] != \\"__pycache__\\" or \\"__pycache__\\" in parts[:-2] or cache is None or source not in path_set or parts[:len(package_prefix)] != package_prefix:\\n                    raise PolicyError(\\"RECORD_BYTECODE\\")\\n                category = \\"excluded_volatile\\"\\n        if category in (\\"owned_regular\\", \\"verified_console_script\\"):\\n            decode_record_hash(row.hash)\\n            decode_record_size(row.size)\\n        elif row.path != own_record:\\n            if bool(row.hash) != bool(row.size):\\n                raise PolicyError(\\"RECORD_HASH_SIZE_PAIR\\")\\n            if row.hash:\\n                decode_record_hash(row.hash)\\n                decode_record_size(row.size)\\n        classified.append((row, category))\\n    if script_names != set(declarations):\\n        raise PolicyError(\\"CONSOLE_RECORD_COVERAGE\\")\\n    if declarations and distribution[\\"dist_info\\"] + \\"/entry_points.txt\\" not in paths:\\n        raise PolicyError(\\"CONSOLE_ENTRY_POINTS\\")\\n    return classified\\n\\n\\ndef payload_required(distribution, path, category):\\n    if category == \\"verified_console_script\\" or path == distribution[\\"dist_info\\"] + \\"/RECORD\\":\\n        return True\\n    parts = tuple(path.split(\\"/\\"))\\n    prefix = OWNED_PREFIXES[distribution[\\"name\\"]]\\n    return distribution[\\"name\\"] in HARNESS_DISTRIBUTIONS and parts[:len(prefix)] == prefix and category == \\"owned_regular\\" and posixpath.splitext(path)[1] in SOURCE_SUFFIXES\\n\\n\\ndef derive_distribution_rows(distribution, interpreter, record_bytes, observation_list, blobs):\\n    classified = classify_record_rows(distribution, interpreter, record_bytes)\\n    observations = {}\\n    for observation in observation_list:\\n        if observation[\\"distribution\\"] != distribution[\\"name\\"]:\\n            raise PolicyError(\\"OBSERVATION_DISTRIBUTION\\")\\n        if observation[\\"path\\"] in observations:\\n            raise PolicyError(\\"OBSERVATION_DUPLICATE\\")\\n        observations[observation[\\"path\\"]] = observation\\n    if set(observations) != {row.path for row, _category in classified}:\\n        raise PolicyError(\\"OBSERVATION_RECORD_COVERAGE\\")\\n    byte_verified, digest_only = 0, 0\\n    categories = {}\\n    for row, category in classified:\\n        observation = observations[row.path]\\n        if observation[\\"kind\\"] != \\"REGULAR\\" or type(observation[\\"size\\"]) is not int or not 0 <= observation[\\"size\\"] <= MAX_OBSERVED_SIZE or type(observation[\\"links\\"]) is not int or observation[\\"links\\"] < 1 or re.fullmatch(r\\"[0-9a-f]{64}\\", observation[\\"sha256\\"]) is None:\\n            raise PolicyError(\\"OBSERVATION_REGULAR\\")\\n        if row.hash and decode_record_hash(row.hash) != observation[\\"sha256\\"]:\\n            raise PolicyError(\\"RECORD_HASH_MISMATCH\\")\\n        if row.size and decode_record_size(row.size) != observation[\\"size\\"]:\\n            raise PolicyError(\\"RECORD_SIZE_MISMATCH\\")\\n        if category == \\"verified_console_script\\" and observation[\\"links\\"] != 1:\\n            raise PolicyError(\\"CONSOLE_HARDLINK\\")\\n        payload = observation[\\"payload\\"]\\n        required = payload_required(distribution, row.path, category)\\n        if required and payload is None:\\n            raise PolicyError(\\"PAYLOAD_REQUIRED\\")\\n        if row.path.startswith(distribution[\\"dist_info\\"] + \\"/\\") and row.path != distribution[\\"dist_info\\"] + \\"/RECORD\\" and payload is not None:\\n            raise PolicyError(\\"METADATA_PAYLOAD_FORBIDDEN\\")\\n        if category == \\"excluded_volatile\\" and row.path != distribution[\\"dist_info\\"] + \\"/RECORD\\" and payload is not None:\\n            raise PolicyError(\\"VOLATILE_PAYLOAD_FORBIDDEN\\")\\n        if payload is not None:\\n            if payload != observation[\\"sha256\\"] or payload not in blobs:\\n                raise PolicyError(\\"PAYLOAD_REFERENCE\\")\\n            data = blobs[payload]\\n            if hashlib.sha256(data).hexdigest() != payload or len(data) != observation[\\"size\\"]:\\n                raise PolicyError(\\"PAYLOAD_BYTES\\")\\n            byte_verified += 1\\n        else:\\n            digest_only += 1\\n        if row.path == distribution[\\"dist_info\\"] + \\"/RECORD\\" and (payload != hashlib.sha256(record_bytes).hexdigest() or blobs[payload] != record_bytes):\\n            raise PolicyError(\\"RECORD_PAYLOAD\\")\\n        if row.path == distribution[\\"dist_info\\"] + \\"/METADATA\\" and (observation[\\"sha256\\"] != distribution[\\"metadata\\"][\\"raw_sha256\\"] or observation[\\"size\\"] != distribution[\\"metadata\\"][\\"raw_size\\"]):\\n            raise PolicyError(\\"METADATA_OBSERVATION\\")\\n        if row.path == distribution[\\"dist_info\\"] + \\"/direct_url.json\\" and (not distribution[\\"direct_url\\"][\\"present\\"] or observation[\\"sha256\\"] != distribution[\\"direct_url\\"][\\"sha256\\"] or observation[\\"size\\"] != distribution[\\"direct_url\\"][\\"size\\"]):\\n            raise PolicyError(\\"DIRECT_URL_OBSERVATION\\")\\n        categories[row.path] = category\\n    has_direct_url = distribution[\\"dist_info\\"] + \\"/direct_url.json\\" in observations\\n    if has_direct_url != distribution[\\"direct_url\\"][\\"present\\"]:\\n        raise PolicyError(\\"DIRECT_URL_OBSERVATION\\")\\n    return {\\"categories\\": categories, \\"record_coverage_complete\\": True,\\n            \\"byte_verified_observation_count\\": byte_verified,\\n            \\"digest_only_observation_count\\": digest_only}\\n\\n\\ndef validate_unique_console_ownership(distributions, interpreter):\\n    owners = set()\\n    for distribution in distributions:\\n        validate_console_scripts(distribution[\\"console_scripts\\"])\\n        for declaration in distribution[\\"console_scripts\\"]:\\n            identity = (interpreter[\\"scripts_directory\\"], declaration[\\"name\\"].casefold())\\n            if identity in owners:\\n                raise PolicyError(\\"CONSOLE_OWNERSHIP\\")\\n            owners.add(identity)\\n", "tools/h23_v4/schema.py": "\\"\\"\\"Strict, bounded JSON and manifest shape for the v4 observation protocol.\\"\\"\\"\\n\\nimport json\\nimport re\\nfrom datetime import datetime\\n\\nPROTOCOL = \\"H23_CAPTURE_PROTOCOL_V4\\"\\nMAX_MANIFEST_BYTES = 8 * 1024 * 1024\\nMAX_JSON_DEPTH = 12\\nMAX_STRING_LENGTH = 4096\\nMAX_LIST_LENGTH = 100_000\\nMAX_JSON_NODES = 500_000\\nSHA256_RE = re.compile(r\\"[0-9a-f]{64}\\\\Z\\")\\n\\n\\nclass PolicyError(ValueError):\\n    \\"\\"\\"A bounded reason code, never untrusted diagnostic text.\\"\\"\\"\\n\\n    def __init__(self, code):\\n        self.code = code\\n        super().__init__(code)\\n\\n\\ndef _pairs(pairs):\\n    result = {}\\n    for key, value in pairs:\\n        if key in result:\\n            raise PolicyError(\\"JSON_DUPLICATE_KEY\\")\\n        result[key] = value\\n    return result\\n\\n\\ndef _nonfinite(_value):\\n    raise PolicyError(\\"JSON_NONFINITE\\")\\n\\n\\ndef _bounded(value, depth=0, budget=None):\\n    if budget is None:\\n        budget = [MAX_JSON_NODES]\\n    budget[0] -= 1\\n    if budget[0] < 0 or depth > MAX_JSON_DEPTH:\\n        raise PolicyError(\\"JSON_LIMIT\\")\\n    if isinstance(value, str):\\n        if len(value) > MAX_STRING_LENGTH or \\"\\\\x00\\" in value or any(0xD800 <= ord(char) <= 0xDFFF for char in value):\\n            raise PolicyError(\\"JSON_LIMIT\\")\\n    elif isinstance(value, dict):\\n        if len(value) > MAX_LIST_LENGTH:\\n            raise PolicyError(\\"JSON_LIMIT\\")\\n        for key, item in value.items():\\n            _bounded(key, depth + 1, budget)\\n            _bounded(item, depth + 1, budget)\\n    elif isinstance(value, list):\\n        if len(value) > MAX_LIST_LENGTH:\\n            raise PolicyError(\\"JSON_LIMIT\\")\\n        for item in value:\\n            _bounded(item, depth + 1, budget)\\n    elif value is not None and type(value) not in (int, bool):\\n        raise PolicyError(\\"JSON_TYPE\\")\\n\\n\\ndef strict_json(raw):\\n    if not isinstance(raw, bytes) or len(raw) > MAX_MANIFEST_BYTES:\\n        raise PolicyError(\\"JSON_LIMIT\\")\\n    try:\\n        value = json.loads(raw.decode(\\"utf-8\\"), object_pairs_hook=_pairs,\\n                           parse_constant=_nonfinite)\\n    except PolicyError:\\n        raise\\n    except ValueError:\\n        raise PolicyError(\\"JSON_SYNTAX\\") from None\\n    except (UnicodeDecodeError, json.JSONDecodeError, RecursionError, OverflowError):\\n        raise PolicyError(\\"JSON_SYNTAX\\") from None\\n    _bounded(value)\\n    return value\\n\\n\\ndef exact_fields(value, required, optional=()):\\n    if not isinstance(value, dict) or not set(required) <= value.keys() or not value.keys() <= set(required) | set(optional):\\n        raise PolicyError(\\"SCHEMA_FIELDS\\")\\n\\n\\ndef string(value, *, pattern=None):\\n    if type(value) is not str or not value or len(value) > MAX_STRING_LENGTH or any(ord(c) < 32 or ord(c) == 127 for c in value):\\n        raise PolicyError(\\"SCHEMA_STRING\\")\\n    if pattern is not None and re.fullmatch(pattern, value) is None:\\n        raise PolicyError(\\"SCHEMA_STRING\\")\\n\\n\\ndef sha256(value):\\n    string(value)\\n    if SHA256_RE.fullmatch(value) is None:\\n        raise PolicyError(\\"SCHEMA_SHA256\\")\\n\\n\\ndef integer(value, minimum=0, maximum=2**63 - 1):\\n    if type(value) is not int or not minimum <= value <= maximum:\\n        raise PolicyError(\\"SCHEMA_INTEGER\\")\\n\\n\\ndef boolean(value):\\n    if type(value) is not bool:\\n        raise PolicyError(\\"SCHEMA_BOOLEAN\\")\\n\\n\\ndef sequence(value, maximum=MAX_LIST_LENGTH):\\n    if not isinstance(value, list) or len(value) > maximum:\\n        raise PolicyError(\\"SCHEMA_LIST\\")\\n\\n\\ndef absolute_path(value):\\n    string(value)\\n    if not value.startswith(\\"/\\") or \\"\\\\\\\\\\" in value or value.endswith(\\"/\\") or any(part in (\\"\\", \\".\\", \\"..\\") for part in value[1:].split(\\"/\\")):\\n        raise PolicyError(\\"SCHEMA_PATH\\")\\n\\n\\ndef _nullable_hash(value):\\n    if value is not None:\\n        sha256(value)\\n\\n\\ndef validate_manifest(manifest):\\n    # Local import avoids a metadata/schema module dependency cycle.\\n    from tools.h23_v4.metadata_policy import (validate_metadata_observation,\\n                                             validate_console_scripts,\\n                                             validate_wheel_observation_identity)\\n\\n    _bounded(manifest)\\n    exact_fields(manifest, (\\"protocol\\", \\"captured_utc\\", \\"capture_program\\", \\"interpreter\\", \\"bootstrap\\", \\"wheels\\", \\"distributions\\", \\"record_evidence\\", \\"file_observations\\", \\"payload_references\\"))\\n    if manifest[\\"protocol\\"] != PROTOCOL:\\n        raise PolicyError(\\"PROTOCOL_UNSUPPORTED\\")\\n    string(manifest[\\"captured_utc\\"], pattern=r\\"[0-9]{8}T[0-9]{6}Z\\")\\n    try:\\n        datetime.strptime(manifest[\\"captured_utc\\"], \\"%Y%m%dT%H%M%SZ\\")\\n    except ValueError:\\n        raise PolicyError(\\"SCHEMA_TIMESTAMP\\") from None\\n    program = manifest[\\"capture_program\\"]\\n    exact_fields(program, (\\"sha256\\", \\"notebook_sha256\\"))\\n    sha256(program[\\"sha256\\"])\\n    _nullable_hash(program[\\"notebook_sha256\\"])\\n    interpreter = manifest[\\"interpreter\\"]\\n    exact_fields(interpreter, (\\"python_version\\", \\"scheme\\", \\"prefix\\", \\"site_packages\\", \\"scripts_directory\\", \\"executable\\"))\\n    sequence(interpreter[\\"python_version\\"], 3)\\n    if len(interpreter[\\"python_version\\"]) != 3:\\n        raise PolicyError(\\"SCHEMA_PYTHON_VERSION\\")\\n    for number in interpreter[\\"python_version\\"]:\\n        integer(number, 0, 255)\\n    if interpreter[\\"python_version\\"][0] != 3 or interpreter[\\"scheme\\"] not in (\\"posix_prefix\\", \\"posix_venv\\", \\"posix_local\\"):\\n        raise PolicyError(\\"SCHEMA_INTERPRETER\\")\\n    absolute_path(interpreter[\\"prefix\\"])\\n    absolute_path(interpreter[\\"scripts_directory\\"])\\n    absolute_path(interpreter[\\"executable\\"])\\n    sequence(interpreter[\\"site_packages\\"], 4)\\n    if not interpreter[\\"site_packages\\"]:\\n        raise PolicyError(\\"SCHEMA_INTERPRETER\\")\\n    for root in interpreter[\\"site_packages\\"]:\\n        absolute_path(root)\\n    if len(set(interpreter[\\"site_packages\\"])) != len(interpreter[\\"site_packages\\"]):\\n        raise PolicyError(\\"SCHEMA_INTERPRETER\\")\\n    bootstrap = manifest[\\"bootstrap\\"]\\n    exact_fields(bootstrap, (\\"recipe_id\\", \\"wheel_references\\", \\"argv\\", \\"executed\\", \\"return_code\\", \\"stdout_byte_count\\", \\"stderr_byte_count\\", \\"stdout_sha256\\", \\"stderr_sha256\\", \\"diagnostic\\"))\\n    if bootstrap[\\"recipe_id\\"] not in (\\"OFFICIAL_NOTEBOOK_CELL_2\\", \\"NOT_RUN\\") or bootstrap[\\"diagnostic\\"] not in (\\"NOT_RUN\\", \\"EXIT_ZERO\\", \\"EXIT_NONZERO\\", \\"EXECUTION_ERROR\\"):\\n        raise PolicyError(\\"SCHEMA_BOOTSTRAP\\")\\n    sequence(bootstrap[\\"wheel_references\\"], 256)\\n    sequence(bootstrap[\\"argv\\"], 300)\\n    for value in bootstrap[\\"wheel_references\\"] + bootstrap[\\"argv\\"]:\\n        string(value)\\n    boolean(bootstrap[\\"executed\\"])\\n    if bootstrap[\\"return_code\\"] is not None:\\n        integer(bootstrap[\\"return_code\\"], -2**31, 2**31 - 1)\\n    for field in (\\"stdout_byte_count\\", \\"stderr_byte_count\\"):\\n        integer(bootstrap[field], 0, 2**40)\\n    for field in (\\"stdout_sha256\\", \\"stderr_sha256\\"):\\n        _nullable_hash(bootstrap[field])\\n    sequence(manifest[\\"wheels\\"], 256)\\n    wheel_names = set()\\n    for wheel in manifest[\\"wheels\\"]:\\n        exact_fields(wheel, (\\"dataset_filename\\", \\"installation_filename\\", \\"size\\", \\"sha256\\", \\"metadata\\"))\\n        for field in (\\"dataset_filename\\", \\"installation_filename\\"):\\n            string(wheel[field], pattern=r\\"[A-Za-z0-9][A-Za-z0-9_.+-]*\\\\.whl\\")\\n        if wheel[\\"dataset_filename\\"] in wheel_names:\\n            raise PolicyError(\\"SCHEMA_DUPLICATE_WHEEL\\")\\n        wheel_names.add(wheel[\\"dataset_filename\\"])\\n        integer(wheel[\\"size\\"], 1, 16 * 2**30)\\n        sha256(wheel[\\"sha256\\"])\\n        validate_wheel_observation_identity(wheel[\\"dataset_filename\\"],\\n                                            wheel[\\"installation_filename\\"], wheel[\\"metadata\\"])\\n    sequence(manifest[\\"distributions\\"], 8)\\n    target_names = {\\"swegemma\\", \\"adk-submission\\", \\"adk-eval-core\\", \\"google-adk\\", \\"google-genai\\", \\"litellm\\", \\"vllm\\", \\"transformers\\"}\\n    distribution_names = set()\\n    for distribution in manifest[\\"distributions\\"]:\\n        exact_fields(distribution, (\\"name\\", \\"version\\", \\"installation_root\\", \\"dist_info\\", \\"metadata\\", \\"direct_url\\", \\"console_scripts\\"))\\n        name = distribution[\\"name\\"]\\n        if type(name) is not str or name not in target_names or name in distribution_names:\\n            raise PolicyError(\\"SCHEMA_DISTRIBUTION\\")\\n        distribution_names.add(name)\\n        string(distribution[\\"version\\"])\\n        absolute_path(distribution[\\"installation_root\\"])\\n        string(distribution[\\"dist_info\\"], pattern=r\\"[A-Za-z0-9_][A-Za-z0-9_.+-]*\\\\.dist-info\\")\\n        validate_metadata_observation(distribution[\\"metadata\\"], name, distribution[\\"version\\"])\\n        direct = distribution[\\"direct_url\\"]\\n        exact_fields(direct, (\\"present\\", \\"sha256\\", \\"size\\", \\"type\\"))\\n        boolean(direct[\\"present\\"])\\n        if direct[\\"present\\"]:\\n            if direct[\\"type\\"] != \\"REGULAR\\":\\n                raise PolicyError(\\"SCHEMA_DIRECT_URL\\")\\n            sha256(direct[\\"sha256\\"])\\n            integer(direct[\\"size\\"], 0, 512 * 1024)\\n        elif direct[\\"type\\"] != \\"ABSENT\\" or direct[\\"sha256\\"] is not None or direct[\\"size\\"] is not None:\\n            raise PolicyError(\\"SCHEMA_DIRECT_URL\\")\\n        validate_console_scripts(distribution[\\"console_scripts\\"])\\n    sequence(manifest[\\"record_evidence\\"], 8)\\n    records = set()\\n    for record in manifest[\\"record_evidence\\"]:\\n        exact_fields(record, (\\"distribution\\", \\"blob\\"))\\n        if type(record[\\"distribution\\"]) is not str or record[\\"distribution\\"] not in distribution_names or record[\\"distribution\\"] in records:\\n            raise PolicyError(\\"SCHEMA_RECORD_REFERENCE\\")\\n        records.add(record[\\"distribution\\"])\\n        sha256(record[\\"blob\\"])\\n    if records != distribution_names:\\n        raise PolicyError(\\"SCHEMA_RECORD_REFERENCE\\")\\n    sequence(manifest[\\"file_observations\\"])\\n    observation_keys = set()\\n    for observation in manifest[\\"file_observations\\"]:\\n        exact_fields(observation, (\\"distribution\\", \\"path\\", \\"kind\\", \\"size\\", \\"sha256\\", \\"payload\\", \\"links\\"))\\n        if type(observation[\\"distribution\\"]) is not str or observation[\\"distribution\\"] not in distribution_names:\\n            raise PolicyError(\\"SCHEMA_OBSERVATION\\")\\n        string(observation[\\"path\\"])\\n        key = (observation[\\"distribution\\"], observation[\\"path\\"])\\n        if key in observation_keys or observation[\\"kind\\"] != \\"REGULAR\\":\\n            raise PolicyError(\\"SCHEMA_OBSERVATION\\")\\n        observation_keys.add(key)\\n        integer(observation[\\"size\\"], 0, 2 * 2**30)\\n        sha256(observation[\\"sha256\\"])\\n        _nullable_hash(observation[\\"payload\\"])\\n        integer(observation[\\"links\\"], 1, 2**20)\\n    sequence(manifest[\\"payload_references\\"])\\n    for reference in manifest[\\"payload_references\\"]:\\n        sha256(reference)\\n    if manifest[\\"payload_references\\"] != sorted(set(manifest[\\"payload_references\\"])):\\n        raise PolicyError(\\"SCHEMA_PAYLOAD_REFERENCE\\")\\n    return manifest\\n\\n\\ndef canonical_json(value):\\n    _bounded(value)\\n    return (json.dumps(value, ensure_ascii=True, allow_nan=False, sort_keys=True, separators=(\\",\\", \\":\\")) + \\"\\\\n\\").encode(\\"utf-8\\")\\n"}')
H23_RUNTIME_DIRECTORY = pathlib.Path(tempfile.mkdtemp(prefix='h23-v4-runtime-'))
for H23_NAME, H23_CONTENT in H23_RUNTIME_SOURCES.items():
    H23_PATH = H23_RUNTIME_DIRECTORY / H23_NAME
    H23_PATH.parent.mkdir(parents=True, exist_ok=True)
    H23_PATH.write_bytes(H23_CONTENT.encode('utf-8'))
sys.path.insert(0, str(H23_RUNTIME_DIRECTORY))
from tools.h23_v4.capture import CaptureConfig, run_capture
H23_ARCHIVE = run_capture(CaptureConfig())
print(json.dumps({'observation_archive': str(H23_ARCHIVE)}, sort_keys=True))
